# Total Recall: Agent Memory and Harness Engineering

### Building a self-improving agent harness, layer by layer, on a single Oracle AI Database

This notebook builds a complete **agent harness** from the ground up and runs it against one local Oracle
AI Database. By the end you will have an agent that **remembers across sessions**, **grounds itself in the
meaning of your data**, **turns the work it does into reusable skills and scheduled automations**, and **keeps
its context window flat** no matter how long a session runs. Every one of those capabilities lives *inside the database* — and the one piece that isn't data, the chat model itself, runs on **Oracle's own OCI Generative AI**. The whole stack is Oracle, end to end.

The notebook is written to make you fluent in the three engineering disciplines that define serious agent work
today. Each is named the first time it appears, and each gets its own layer:

| Discipline | The question it answers | Where it lives here |
|---|---|---|
| **Memory Engineering** | *What is the agent?* A persistent entity with knowledge that accumulates. | Parts 2, 4, 6 |
| **Context Engineering** | *What does the agent see right now?* The context window as a scarce, managed resource. | Parts 3, 8 |
| **Harness Engineering** | *Everything that is not the model:* the loop, tools, state, memory I/O, budgets, recovery, and outcome tracking. | Parts 6, 7, 8 |

> ### 🎓 This is the **student** notebook
>
> Several implementations have been replaced with numbered **`TODO`** placeholders. Each TODO has a **`docs/todoN.md`** explainer (with a copy-paste solution) and a **`✅ TODO N check`** cell that must pass before you continue. Work top to bottom — the TODO numbers run in reading order. Everything else runs as-is; the full answer key is `total_recall_complete.ipynb`.

> ### The one equation this whole notebook is about
> # `Agent = Model + Harness`
> The model is a **frozen reasoning utility**. It does not learn between calls, it has no memory, and it
> cannot act. The **harness** is everything we build around it: the tools that give it hands, the database
> that gives it memory, the loop that lets it take many steps, the context discipline that keeps it
> affordable, and the checks that tell us whether it actually worked. *Improving an agent is rarely about the
> model. It is about the harness.* This notebook is a harness, built one layer at a time.

![The harness, dissected layer by layer (the model is layer 1; everything else is what you build)](images/agent_harness.png)

## How this notebook is organised, and how to read it

The build is a **story told in Parts**, bottom up. Each Part opens by recapping what the previous Part gave us
and how this one continues the construction, and closes with a **Key takeaway** box. Inside a Part, **every
code cell is preceded by a markdown cell** that explains what the code does and *why it is built that way*.
Read those first. They are the actual lesson. Code cells are kept short on purpose, so each one teaches a
single idea you can run and inspect before moving on.

| Part | Title | What it adds to the agent |
|---|---|---|
| **0** | Orientation | The thesis, the architecture, the design choices |
| **1** | Foundation and Models | A database, a least-privilege user, and the models (one chat LLM out; two ONNX models in) |
| **2** | The Memory Substrate | *Where* memory physically lives: the filesystem-versus-database question, resolved as **both, inside the database** |
| **3** | Encoding and Retrieval | The write path (text to vectors, in-DB, via **OracleVS**) and the read path (the retrieval ladder) |
| **4** | Cognitive Memory | The memory **taxonomy** backed by the Oracle AI Agent Memory package, fact **extraction**, and the scratch-to-long-term **promotion** pipeline |
| **5** | The Semantic Layer | Teaching the agent what *your* schema means, refreshed on a schedule |
| **6** | Skills and Automations | The agent improving itself: the **toolbox**, the **skillbox**, and *continual learning in token space* |
| **7** | The Agent Loop | The graph that drives everything above, with dynamic recall and failure recovery |
| **8** | Context Engineering | Compaction plus offloading, with the window growth **measured**, not simulated |
| **9** | Continuity | A cross-session recall check: a fact written in one session is recalled in a brand-new one |
| **10** | The End-to-End Story | One narrative that exercises every layer |

> 💡 **A note on the build order.** We build **bottom up** (from Part 1 upward) because each layer
> physically depends on the one beneath it. You cannot encode into stores you have not created, retrieve from
> an empty store, or run a loop with no tools or memory. Context Engineering comes late because it *modifies*
> the loop rather than adding a new store.

## The architecture at a glance

![The whole harness as one engine — the reference architecture, bottom-up](images/the_whole_harness_one_engine_ref_architecture.png)

Read this from the bottom up — that is the order we build it. A **user turn** enters the agent loop at the top; the **database foundation** everything rests on is at the bottom. Colour groups the layers: the agent loop (dark), the action path (amber: skills, automations, context), the memory/data path (blue: cognitive memory, retrieval, encoding, semantic layer), and the substrate + foundation (green).

```mermaid
flowchart TB
    U(["user turn"]):::io --> P7

    P7["<b>Part 7 · Agent Loop</b> — a typed state graph<br/>assemble_context → call_model → dispatch_tools → persist<br/>dynamic recall of skills + workflow recipes · failure recovery<br/>loops under an iteration + time budget"]:::loop

    P7 -- "tools — the only way it can act" --> P6
    P7 -- "read / write" --> P4

    P6["<b>Part 6 · Skills and Automations</b><br/>TOOLBOX (tool registry) · SKILLBOX (skill registry)<br/>harvester · automations"]:::action
    P8["<b>Part 8 · Context Engineering</b><br/>compaction (context card)<br/>offloading → table OR file"]:::action
    P4["<b>Part 4 · Cognitive Memory (OAMP)</b><br/>episodic · semantic (extracted) · working<br/>workflow recipes with outcome<br/>scratch → long-term promotion (scheduled)"]:::mem
    P3R["<b>Part 3 · Retrieval — read path</b><br/>keyword · vector · hybrid (RRF)<br/>rerank (cross-encoder)"]:::mem
    P3W["<b>Part 3 · Encoding — write path</b><br/>OracleVS + OracleEmbeddings<br/>embedding computed in-database (provider=database)"]:::mem
    P5["<b>Part 5 · Semantic Layer</b><br/>comments + annotations + constraints + workload<br/>→ a searchable catalog (scheduled)"]:::mem

    P6 <--> P4
    P6 --> P8
    P4 --> P3R
    P3R --> P3W
    P5 <--> P3W

    subgraph P2["Part 2 · Memory Substrate — one ACID / security / backup boundary"]
      direction LR
      P2T["vector and relational tables<br/>(durable, searchable)"]
      P2S["in-DB scratch filesystem<br/>(POSIX-like, SecureFile LOBs)"]
      P2S -. promote .-> P2T
    end

    P3W --> P2T
    P2 --> P1["<b>Part 1 · Foundation</b><br/>the database · a least-privilege AGENT user<br/>models: ONNX embedder + ONNX reranker (in the DB) · chat LLM (OCI GenAI)"]:::base

    classDef loop fill:#1f2937,stroke:#0f172a,color:#ffffff
    classDef action fill:#fde68a,stroke:#b45309,color:#1f2937
    classDef mem fill:#bfdbfe,stroke:#1d4ed8,color:#1f2937
    classDef base fill:#d1fae5,stroke:#047857,color:#1f2937
    classDef io fill:#ffffff,stroke:#6b7280,color:#1f2937
```


**How to read the diagram, the system component each box represents:**

| Layer | Its role in the system | What it physically is |
|---|---|---|
| ⚙️ **Part 1 Foundation** | the ground everything stands on | the database, a least-privilege `AGENT` user, the loaded ONNX models |
| 🗄️ **Part 2 Memory Substrate** | the floor: where bytes live | one ACID boundary, relational/vector tables + an in-DB scratch filesystem |
| 🧬 **Part 3 Encoding** | the write path | OracleVS turns text into in-database vectors |
| 🔎 **Part 3 Retrieval** | the read path | the keyword / vector / hybrid / rerank ladder |
| 🗺️ **Part 5 Semantic Layer** | the map of your data's meaning | a searchable catalog of schema knowledge, refreshed on a schedule |
| ✂️ **Part 8 Context Engineering** | the attention budget | compaction (context card) + offloading keep the window flat |
| 💾 **Part 4 Cognitive Memory** | the memory | OAMP: episodic, semantic, working, and procedural stores |
| 🛠️ **Part 6 Toolbox / Skillbox** | the hands and the playbook | tool registry + skill registry, both retrieved by meaning |
| 🧠 **Part 7 Agent Loop** | the driver that ties it together | a typed state graph that assembles context, calls the model, runs tools |

Two arrows carry the whole story: **tools** (down the left, the only way the agent can *act*) and **read/write**
(down the right, how it *remembers*). Every box lives inside the one Oracle AI Database. The only thing reached over the network is the chat model — and that call goes to **Oracle's own OCI Generative AI**, so even the model is Oracle.

## Design choices (made explicit up front)

A handful of decisions shape everything below. They are stated here so nothing later feels arbitrary.

| Choice | What we picked | Why |
|---|---|---|
| **Scenario** | A retail-analytics agent over a seeded `CUSTOMERS / PRODUCTS / ORDERS / ORDER_ITEMS` schema | Universally understood; analytical questions map cleanly onto scheduled artifacts |
| **All embeddings** | Produced **inside the database** by a loaded ONNX model, reached through **OracleVS** | No text leaves the database to be vectorised; one model, one vector space |
| **Vector store** | The **`langchain-oracledb` `OracleVS`** class with an in-database `OracleEmbeddings` | A maintained, idiomatic store, so our own code stays small and the lesson stays on the ideas |
| **Cognitive memory** | The **Oracle AI Agent Memory** package (OAMP), with an in-database embedder and **fact extraction** | A real memory system of record, not a hand-rolled stand-in |
| **The scratchpad** | An in-database **POSIX-like filesystem** (SecureFile LOBs), exposed to the agent as file tools | File ergonomics *and* database guarantees under one boundary; the agent's short-term memory |
| **The toolbox** | A **tool registry** table: each tool's JSON schema plus an embedding, retrieved top-k by meaning | The prompt does not bloat as the toolset grows; the model sees only the tools that fit the turn |
| **The skillbox** | A **skill registry** table of `SKILL.md` playbooks, retrieved on demand | The Agent Skills pattern, server-side: *continual learning in token space* |
| **Chat model** | **OCI Generative AI** (OpenAI-compatible endpoint), the only outbound call at run time | Oracle powers the model too, so the _whole_ stack stays on Oracle |

![Oracle is the engine, not a step — ingestion, embeddings, vector search and rerank all run in the database](images/oracle_engine.png)


> 💡 **Key term: in-database embeddings.** An *embedding* is a vector that captures the meaning of a piece of
> text, so that closeness in vector space means closeness in meaning. Normally you call an external service to
> produce embeddings. Here we load a small ONNX model **into the database** and compute embeddings with a SQL
> expression. The write path never leaves the engine: nothing to secure in transit, nothing to rate-limit, and
> one consistent vector space for every store. `OracleVS` plus `OracleEmbeddings(provider="database")` gives us
> that property with very little code of our own.

---
# Part 1: Foundation and Models

> **Where we are.** Part 0 laid out the thesis (`Agent = Model + Harness`) and the architecture. We have a
> blueprint but no ground to build on.
>
> **What this part does.** Your Codespace already poured the foundation — it started Oracle AI Database,
> created the least-privilege **`AGENT`** user with exactly the grants the later layers need, sized the
> **vector memory pool**, and loaded the in-database **embedder** (all in `scripts/seed_oracle.py`). So this
> part is short: **connect** to that provisioned harness, meet the **models** that make encoding a SQL
> expression, and wire up the **chat model**. No `docker`, no passwords, no config to edit — just _Run All_.
>
> **Why it matters.** After this part, *encoding text into meaning is a SQL expression*. That single fact —
> embeddings computed in the engine — is what lets every later layer (memory, retrieval, the semantic
> catalog, the toolbox, the skillbox) keep all data and all computation inside one security boundary.

### 1.1 Connect to your provisioned harness

Everything in this notebook runs as the least-privilege **`AGENT`** user your Codespace created. The
plumbing — the config (`CFG`), a resilient connection, and the handful of SQL helpers the rest of the
notebook builds on — lives in **[`workshop_setup.py`](workshop_setup.py)** so it stays out of your way.
Importing it connects you and hands those back. There is **nothing to configure and no password to type**.

> 💡 Curious what's behind it? Open `workshop_setup.py` — it's the connection plus the `fetch_rows` /
> `execute_sql` / `ddl_idempotent` / `create_table` helpers, all read from the environment your Codespace
> set. We keep them in a module so you focus on the **harness**, not the boilerplate.

In [ ]:
import os, json, time, warnings, oracledb
warnings.filterwarnings("ignore", message=".*asynchronous method.*")
warnings.filterwarnings("ignore", message=".*IProgress not found.*")

# Your Codespace already provisioned the harness (AGENT user + grants + the in-database embedder — see
# scripts/seed_oracle.py). Importing workshop_setup connects you as that AGENT user and hands back the
# config (CFG), a live connection (conn), and the SQL helpers the rest of the notebook builds on.
from workshop_setup import (CFG, conn, AGENT, AGENT_PWD, connect_with_retry,
                            fetch_rows, execute_sql, ddl_idempotent, create_table, RERANK_AVAILABLE,
                            make_rotating_chat_model, make_rotating_oamp_llm)

print(f"Connected as {AGENT} to {CFG['DSN']} | model {CFG['LLM_PROVIDER']}/{CFG['LLM_MODEL']} "
      f"| embedder {CFG['EMBED_MODEL']} | reranker {'loaded' if RERANK_AVAILABLE else 'fallback (optional)'}")

### 1.2 The models that live in the database

Two ONNX models do the heavy lifting, and both run **inside** Oracle — so encoding text into meaning is a
SQL expression, not a network round-trip. Your Codespace loaded them; you just use them.

- **The embedder** — an augmented `all_MiniLM_L12_v2` (384-dim). "Augmented" means tokenisation is baked
  into the graph, so `VECTOR_EMBEDDING(model USING 'raw text' AS DATA)` takes plain text and returns a
  vector. This single model serves *every* store in the notebook, so everything shares one 384-dim space
  and there are never dimension mismatches. (The checkpoint in **1.4** proves it is loaded.)
- **The reranker** *(optional)* — a cross-encoder that reads a `(query, document)` pair *together* and
  returns one relevance score, used in Part 3 to reorder results. If it is loaded, `RERANK_AVAILABLE` is
  `True`; otherwise the retrieval ladder simply falls back to hybrid order.

> 💡 **Bi-encoder vs cross-encoder.** The embedder is a **bi-encoder**: it encodes the query and each
> document *separately*, so retrieval is a fast nearest-neighbour lookup (document vectors are precomputed)
> but it can't model word-level interaction. The reranker is a **cross-encoder**: it reads query and
> document *together* — far more accurate, far more expensive — so you only run it on the handful of
> candidates the bi-encoder shortlisted. *Retrieve cheaply, then rerank precisely.*

```mermaid
flowchart LR
    subgraph OFFLINE["Indexing time (once per document)"]
        direction TB
        D["Documents / facts"] --> BE1["BI-ENCODER (embedder)<br/>encodes each document ALONE<br/>→ one fixed vector per doc"]
        BE1 --> VEC[("Vector index<br/>HNSW in Oracle AI DB")]
    end
    subgraph ONLINE["Query time (per question)"]
        direction TB
        Q["User query"] --> BE2["BI-ENCODER (embedder)<br/>encodes the query ALONE<br/>→ one query vector"]
        BE2 -->|"cosine nearest-neighbour<br/>(cheap, approximate)"| VEC
        VEC -->|"top-N candidates"| CE["CROSS-ENCODER (reranker)<br/>reads query + document TOGETHER<br/>→ one relevance score per pair"]
        CE -->|"top-k, precisely ordered"| OUT["Answer context"]
    end
```

### 1.3 The chat model: the reasoning core of the harness

The chat model is the **only** part of this system reached over the network at run time. It runs on **OCI
Generative AI** (through its OpenAI-compatible endpoint), so even the model is Oracle and the whole stack
stays on one vendor. `LLM_PROVIDER` is just an escape hatch: set it to `openai` to use OpenAI instead,
without touching the rest of the notebook.

It is also the **reasoning core** of everything we build, the cognitive engine the harness is wrapped
around. Recall the equation, `Agent = Model + Harness`: this model is the *Model*, and every other layer in
this notebook (memory, retrieval, tools, the loop) is *Harness* built around it. Three properties of that
core decide the entire design that follows:

- **It reasons.** Given a context, it decomposes the task, decides the next step, chooses a tool, and
  synthesises an answer. Turning *a goal plus some context* into *a next action* is the one thing only the
  model can do, which is exactly why it sits at the centre.
- **It is frozen.** Its weights never change between calls; it cannot learn from this conversation or this
  user. Any learning the agent appears to do (Part 6) lives in **tokens we store and re-supply**, never in
  the model itself.
- **It is stateless.** A call is effectively a pure function, context in and reasoning out, with **no memory
  of the previous call and no side effects**. It knows only what is in its context window *right now*; it
  cannot remember, read your database, or run a query until the harness hands it memory and tools.

That last property is why the rest of this notebook exists. A frozen, stateless reasoner is powerful but
blind: it can act only on what you put in front of it, and it forgets everything the instant the call
returns. The harness's whole job is to **assemble the right context for the core** (memory, grounding,
skills, the relevant tools) and to **carry out the decisions it returns** (run the tool, persist the result,
loop). It is also why, in practice, *improving an agent is rarely about swapping the model and almost always
about improving the harness around it*, the core is fixed, so the leverage is in what surrounds it.

In [ ]:
def make_chat_model():
    """The chat LLM — OCI Generative AI via its OpenAI-compatible endpoint, so Oracle powers the model
    too (not just embeddings/memory). LangChain's ChatOpenAI speaks the same API, so the LangGraph loop
    and tool-calling below are unchanged. Set LLM_PROVIDER=openai to use OpenAI directly."""
    from langchain_openai import ChatOpenAI
    if CFG["LLM_PROVIDER"] == "oci":
        return make_rotating_chat_model()
    assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY is not set (export it, or set a Codespaces secret)."
    return ChatOpenAI(model=CFG["LLM_MODEL"])

llm = make_chat_model()
print("Chat model ready:", CFG["LLM_PROVIDER"], "/", CFG["LLM_MODEL"])


> ### 🧪 Your turn: talk to the reasoning core directly
>
> Before we build any harness, feel what the bare core is. Put a question in `QUESTION` below and run the
> cell to call the model **directly**, with no memory of earlier cells, no database, and no tools. It answers
> purely from what it learned in pre-training plus the one message you send it. This is the only place in the
> notebook you talk to the raw model; everywhere else it speaks *through* the harness.
>
> Try a few and watch the boundary between what the core knows and what it cannot reach:
>
> | Try asking… | What it shows |
> |---|---|
> | `In two sentences, what is an agent harness, and why is the model only part of an agent?` | the core **reasoning** unaided, explaining a concept |
> | `Explain Reciprocal Rank Fusion in one short paragraph.` | **general knowledge** baked into the weights |
> | `How many paid orders did we get in the last 30 days?` | the core's **blind spot**: it has no link to your data, so it can only guess or refuse, until the harness (Part 7) lets it actually query the database and answer for real |
>
> That third question failing is the whole point: it previews exactly what the next nine Parts add.

### 🧩 TODO 1 — Talk to the reasoning core

The implementation in the next cell has been removed. Open **[`docs/todo1.md`](docs/todo1.md)** for a guided explanation and the solution snippet, then replace the placeholder.

Run the **`✅ TODO 1 check`** cell right after — it must pass before you continue.

In [ ]:
from langchain_core.messages import HumanMessage

# 🧩 TODO 1 — set QUESTION to anything you want to ask the bare model, then run this cell.
# (No memory, no tools, no database yet — just the reasoning core.) See docs/todo1.md
QUESTION = ""

try:
    answer = llm.invoke([HumanMessage(content=QUESTION)]).content
    print("Q:", QUESTION)
    print("A:", answer)
except Exception as e:
    print("Model call failed:", str(e).splitlines()[0])
    print("Make sure OCI_GENAI_API_KEY is set (see 1.3 above).")


In [ ]:
# ✅ TODO 1 check — do not edit. This must pass before you continue.
assert QUESTION.strip(), "Set QUESTION to a non-empty question, then re-run the cell above."
print("✓ TODO 1 complete — you asked the reasoning core:", repr(QUESTION[:70]))


### 1.4 Checkpoint: the foundation holds

A layer is only "done" when a runnable check proves it in isolation. Here: an in-database embedding returns a
384-dim vector, and the model catalog lists the embedder (and the reranker, if it loaded).

In [ ]:
row = fetch_rows(conn, f"SELECT VECTOR_EMBEDDING({CFG['EMBED_MODEL']} USING 'hello world' AS DATA) AS v FROM dual")
dim = len(row[0]["V"]) if row and row[0]["V"] is not None else 0
print("embedding dim:", dim, "(expected", CFG["VECTOR_DIM"], ")")
print("models:", [m["MODEL_NAME"] for m in fetch_rows(conn, "SELECT model_name FROM user_mining_models ORDER BY 1")])
assert dim == CFG["VECTOR_DIM"], "Embedding model not loaded correctly; fix Part 1 before continuing."
print("\nPART 1 OK.")

> ### ✅ Key takeaway, Part 1
> The agent now has **ground to stand on**: a least-privilege identity, a sized vector pool, and, the part that
> shapes everything else, **models living inside the database**. Producing an embedding is now a SQL
> expression, so every store we build next can be filled and searched without any text ever leaving the
> engine. Next, in **Part 2**, we decide *where the agent's memory physically lives* before we write a single
> byte to it.

---
# Part 2: The Memory Substrate

> **Where we are.** Part 1 gave us a database, an identity, and in-database models. Turning text into meaning
> is now a SQL expression.
>
> **What this part does.** Before writing a single byte of memory, we decide *where the agent's memory
> physically lives*. This is one of the most-debated questions in agent engineering, **filesystem or
> database?**, and we resolve it the way strong teams now do: **both, in the right layers, inside one
> database**.
>
> **Why it matters.** The substrate is the contract every later layer obeys. Getting it right means the agent
> gets file ergonomics *and* transactional guarantees under a single security, backup, and audit boundary, with
> no second system to secure or reconcile.

![The Oracle Database file system — a filesystem interface agents already speak, backed by the database](images/filesystem_database.png)

### Why so many agents reach for the filesystem as memory

Look at how production agents are built and one pattern stands out: many of them lean on **plain files** as
memory. Scratch notes, a running `NOTES.md`, transcripts written to disk. This is not laziness. There are
sound reasons, and naming them tells us what our substrate must preserve.

- **It matches what the model already knows.** Filesystems and shell commands are among the most common things
  in an LLM's training data. A model handles `read_file`, `list_files`, and `grep` natively, so a file
  interface costs almost no instructions to teach. A bespoke memory API has to be *explained* in the prompt,
  and every token spent explaining a tool is a token not spent on the task.
- **One interface over many stores.** The old Unix habit of treating every resource as something you read,
  write, and list lets heterogeneous memory (notes, transcripts, retrieved documents, tool output) be reached
  the same way. One mental model, many backing stores.
- **It externalises state cheaply.** Writing a result to a file and keeping only a short reference in context
  is the simplest possible form of *context offloading* (Part 8). The file is the overflow store.

The other half of the story is the catch. Naive files have **no concurrency safety** (two writers can corrupt
each other), **no search beyond grep**, and they sit **outside** the database's backups and access controls.
The sensible conclusion is not "files versus database." It is **a file-like interface for what the agent
sees, and durable database storage for what persists.** Our substrate delivers exactly that, and it puts the
filesystem **inside** the database, so we never trade away durability, search, or security to get the file
ergonomics.

### The routing policy: what goes where (the contract every later layer obeys)

| Data | Substrate | Why |
|---|---|---|
| Durable facts, entities, workflow recipes, skills, tool metadata, summaries | **Vector / relational tables** | Searched by meaning, kept for the long term; ACID plus hybrid search |
| Conversation (the episodic record) | **OAMP tables** (Part 4) | Threaded, queryable, consolidated |
| Working scratchpad: intermediate results, drafts, documents being read | **In-database files** under `/scratch` | Whole-document reads, progressive disclosure, cheap writes, no premature embedding |
| A scratchpad artifact worth keeping | **Promote** file to long-term memory (Part 4) | One pipeline; the bridge that joins the two halves |

We now build the **file half** of this substrate. The **table half** is built in Part 3, and Part 4 connects
them with a scheduled promotion pipeline.

### 2.1 Why a SecureFile-LOB table (and not Oracle's native DBFS)

Oracle ships a real POSIX filesystem that stores files as rows with SecureFile LOBs *inside* the database
(`CREATEFILESYSTEM` then `REGISTERSTORE` then `MOUNTSTORE`). Mounting it needs `CREATE TABLESPACE` and DBFS
privileges that a least-privilege user does not have. Rather than fight those privileges, we build the agent's
filesystem on **the very same storage primitive** ourselves: a single table whose content column is a
SecureFile LOB. The next cells do exactly that. The ergonomics and the guarantees (durability, ACID, backups)
are identical; we simply skip the privileged mount.

### 2.2 The scratch table behind the filesystem

Here is the heart of Part 2. We back the agent's filesystem with a single table whose content column is a
**SecureFile LOB**, the exact primitive the native filesystem uses under the covers. Every file is a row, and
every write rides a normal database transaction. We start with the table itself.

> 💡 **Key term: SecureFile LOB.** A *LOB* (Large OBject) is Oracle's column type for big text or binary
> values; *SecureFile* is its modern, high-performance variant that stores those values efficiently inside the
> database (with optional compression and deduplication). It is what lets a single table column hold an entire
> file's contents while still behaving like ordinary, transactional table data.

### 🧩 TODO 2 — Design the scratch table (SecureFile LOB)

The implementation in the next cell has been removed. Open **[`docs/todo2.md`](docs/todo2.md)** for a guided explanation and the solution snippet, then replace the placeholder.

Run the **`✅ TODO 2 check`** cell right after — it must pass before you continue.

In [ ]:
# 🧩 TODO 2 — complete the CREATE TABLE so the filesystem has a place to store files.
# Give agent_scratch: a BLOB 'content' column (the file body), 'is_dir' and 'promoted' CHAR(1) flags
# (DEFAULT 'N'), an 'updated_at' TIMESTAMP, and store the content LOB AS SECUREFILE. See docs/todo2.md
SCRATCH_DDL = '''CREATE TABLE agent_scratch (
  path        VARCHAR2(400) PRIMARY KEY
  -- ...add the remaining columns and the SECUREFILE LOB clause here...
)'''
create_table(conn, SCRATCH_DDL)
print("agent_scratch table ready (SecureFile LOB content; one row per file).")


In [ ]:
# ✅ TODO 2 check — do not edit. This must pass before you continue.
_ddl = SCRATCH_DDL.upper()
for _kw in ("CONTENT", "BLOB", "IS_DIR", "PROMOTED", "UPDATED_AT", "SECUREFILE"):
    assert _kw in _ddl, f"your CREATE TABLE is still missing {_kw} — re-read the TODO and docs"
_cols = {c["COLUMN_NAME"] for c in fetch_rows(conn,
        "SELECT column_name FROM user_tab_columns WHERE table_name='AGENT_SCRATCH'")}
_need = {"PATH", "CONTENT", "IS_DIR", "PROMOTED", "UPDATED_AT"}
assert _need <= _cols, f"agent_scratch is missing columns: {_need - _cols}"
print("✓ TODO 2 complete — the scratch table holds one row per file, content in a LOB.")


### 2.2b A POSIX-like filesystem class over that table

`ScratchFS` wraps the SecureFile-LOB table in ordinary file operations (`mkdir`, `write`, `append`, `read`,
`exists`, `list`). We use a *POSIX-like* shape on purpose: the model has seen millions of file examples in
training, so file tools cost almost no explanation in the prompt. And because each operation is just a row
op, the files are **ACID**, survive process death, and ride the database's backups and access control —
none of which a file on the host's disk gives you. *(Full walkthrough + the SQL: see the TODO guide.)*

### 🧩 TODO 3 — The in-database scratch filesystem (write & read)

The implementation in the next cell has been removed. Open **[`docs/todo3.md`](docs/todo3.md)** for a guided explanation and the solution snippet, then replace the placeholder.

Run the **`✅ TODO 3 check`** cell right after — it must pass before you continue.

In [ ]:
class ScratchFS:
    "A POSIX-like filesystem inside the database (one row per file, content in a SecureFile LOB)."
    def __init__(self, conn, mount="/scratch"):
        self.conn = conn
        self.mount = mount.rstrip("/")

    def _abs(self, path):
        path = "/" + path.strip("/")
        return path if path.startswith(self.mount) else self.mount + path

    def mkdir(self, path):
        p = self._abs(path)
        execute_sql(self.conn, '''MERGE INTO agent_scratch d USING (SELECT :p AS path FROM dual) s ON (d.path = s.path)
                        WHEN NOT MATCHED THEN INSERT (path, is_dir) VALUES (:p, 'Y')''', {"p": p})

    def write(self, path, content):
        # 🧩 TODO 3 — store the file in the database (UPSERT one row of agent_scratch). See docs/todo3.md
        raise NotImplementedError("Complete TODO 3 — open docs/todo3.md")

    def append(self, path, content):
        self.write(path, (self.read(path) if self.exists(path) else "") + content)

    def read(self, path):
        # 🧩 TODO 3 — read the file's content back out of the database. See docs/todo3.md
        raise NotImplementedError("Complete TODO 3 — open docs/todo3.md")

    def exists(self, path):
        return bool(fetch_rows(self.conn, "SELECT 1 FROM agent_scratch WHERE path = :p", {"p": self._abs(path)}))

    def list(self, path="/"):
        pre = self._abs(path).rstrip("/") + "/%"
        rows = fetch_rows(self.conn, "SELECT path FROM agent_scratch WHERE path LIKE :pre AND is_dir = 'N' ORDER BY path",
                          {"pre": pre})
        return [r["PATH"] for r in rows]

print("ScratchFS class ready.")


In [ ]:
# ✅ TODO 3 check — do not edit. This must pass before you continue.
_fs = ScratchFS(conn, mount="/scratch")
_fs.write("/todo1.txt", "total recall")
assert _fs.exists("/todo1.txt"), "after write(), the file should exist in the database"
assert _fs.read("/todo1.txt") == "total recall", "read() must return exactly what write() stored"
_fs.write("/todo1.txt", "updated")
assert _fs.read("/todo1.txt") == "updated", "write() to an existing path should overwrite it"
print("✓ TODO 3 complete — the agent has an ACID scratch filesystem that lives inside the database.")


### 2.2c Mount it and create the working directories

We instantiate the filesystem and lay out the directories the agent will use: reports, notes, tool output, and
an inbox for documents that arrive to be read.

In [ ]:
fs = ScratchFS(conn, mount=CFG["DBFS_MOUNT"])
for d in ("/reports", "/notes", "/tool_out", "/inbox"):
    fs.mkdir(d)
print("scratch filesystem ready (SecureFile-LOB backed). Dirs: /reports /notes /tool_out /inbox")

### 2.3 File tools the agent can call (its short-term memory)

These are the functions the *agent* calls to use its scratchpad as **short-term memory** — jot an
intermediate result, draft a report, stash a document mid-task — without committing to long-term memory yet
(we register them in the toolbox in Part 6). The reading tools (`read_file`, `tail_file`, `grep_files`)
embody **progressive disclosure**: hand the model a small, relevant slice of a large store, never the whole
thing — the file-substrate twin of vector retrieval.

### 🧩 TODO 4 — Grep the agent's scratch memory

The implementation in the next cell has been removed. Open **[`docs/todo4.md`](docs/todo4.md)** for a guided explanation and the solution snippet, then replace the placeholder.

Run the **`✅ TODO 4 check`** cell right after — it must pass before you continue.

In [ ]:
def write_file(path, content):
    """Agent tool: create or overwrite a scratch file (the agent's short-term memory)."""
    fs.write(path, content)
    return {"written": fs._abs(path), "bytes": len(content)}

def append_file(path, content):
    """Agent tool: append to a scratch file, creating it if needed."""
    fs.append(path, content)
    return {"appended": fs._abs(path)}

def read_file(path):
    return fs.read(path)

def tail_file(path, n=20):
    return "\n".join(fs.read(path).splitlines()[-n:])

def grep_files(pattern, root="/", ignorecase=True):
    # 🧩 TODO 4 — search every scratch file line-by-line for a regex; return {path, line, text} hits. See docs/todo4.md
    raise NotImplementedError("Complete TODO 4 — open docs/todo4.md")

def list_files(root="/"):
    return fs.list(root)

print("File tools ready: write_file, append_file, read_file, tail_file, grep_files, list_files")


In [ ]:
# ✅ TODO 4 check — do not edit.
write_file("/notes/risk.md", "line one\nsupplier risk is high\nline three")
_hits = grep_files("supplier risk")
assert isinstance(_hits, list) and _hits, "grep_files should return a non-empty list of {path, line, text} hits"
_mine = [h for h in _hits if h["path"].endswith("/risk.md")]
assert _mine, "the file we just wrote should appear in the results"
assert _mine[0]["line"] == 2 and "supplier risk" in _mine[0]["text"].lower(), "report the matching line and its 1-based number"
print("✓ TODO 4 complete — the agent can grep its own in-database scratch memory.")


### 2.4 Checkpoint: the filesystem lives in the database

Two things to prove. First, the three reading modes work (full, tail, grep), which is progressive disclosure in
action. Second, and more important, **durability across sessions**.

The second check deserves attention, because it is the whole point. 

We open `conn2`, a **completely separate database connection**, build a *fresh* `ScratchFS` object on it, and read the same file back. 

Think about what that means. 

A file written to the operating system's disk would be invisible to a new database connection;
the connection talks to the database, not to the kernel's filesystem. 

Our file is different: it is a *row in a table inside the database*, so any session that can see the table can see the file.

That is exactly the property that turns a scratchpad into real memory rather than a per-process temporary buffer, and it is why,
later, a brand-new agent session can pick up notes an earlier session left behind. We then close `conn2` to
show the data outlives the connection that read it.

In [ ]:
doc = '''# Q3 Pipeline Review

Revenue grew 12% QoQ, led by the Outdoors category.
Risk: a single supplier accounts for 40% of Outdoors COGS.
Decision: dual-source by Q1; rationale = concentration risk, not price.
'''
fs.write("/reports/q3_review.md", doc)
print("--- full read ---");      print(read_file("/reports/q3_review.md")[:90], "...")
print("--- tail (2 lines) ---"); print(tail_file("/reports/q3_review.md", 2))
print("--- grep 'risk' ---");    print(grep_files("risk", "/reports"))

In [ ]:
# A second, independent connection cannot see kernel files from this process, but it CAN see this one,
# because the file is a row in a table inside the database.
conn2 = connect_with_retry(AGENT, AGENT_PWD, CFG["DSN"])
survived = ScratchFS(conn2, mount=CFG["DBFS_MOUNT"]).read("/reports/q3_review.md")
conn2.close()
assert "Revenue grew 12%" in survived, "scratch file did not survive a new session!"
print("\nPART 2 OK: the file survived a brand-new database session; it lives in the DB, not the kernel.")

### 2.5 Why "in the database" matters: an ACID demonstration

The previous cell showed **durability**. This one shows **isolation**, the property that actually bites agents:
when two tool calls run at the same time, do their writes corrupt each other? We run the classic *lost-update*
race. Two workers each increment a counter 500 times. Against a plain file with no locking, the
read-modify-write steps interleave and updates are silently lost. Against the database, a single atomic
`UPDATE ... SET n = n + 1` takes a row lock, so every increment counts. Filesystems are not broken for
concurrency; they just need you to remember to lock. The database is safe by default.

In [ ]:
import threading, tempfile, time as _t

# (A) plain OS file, read-modify-write with NO lock -> updates are lost when threads interleave
cf = os.path.join(tempfile.gettempdir(), "tr_counter.txt")
open(cf, "w").write("0")
def os_inc():
    for _ in range(500):
        v = int(open(cf).read() or 0)
        _t.sleep(0.0002)                 # widen the race window so the effect is visible and reliable
        open(cf, "w").write(str(v + 1))
ts = [threading.Thread(target=os_inc) for _ in range(2)]
[t.start() for t in ts]; [t.join() for t in ts]
os_final = int(open(cf).read())

In [ ]:
# (B) database counter, atomic UPDATE -> row-locked, isolated
create_table(conn, "CREATE TABLE race_counter (id NUMBER PRIMARY KEY, n NUMBER)")
execute_sql(conn, "DELETE FROM race_counter"); execute_sql(conn, "INSERT INTO race_counter VALUES (1, 0)")
def db_inc():
    c = connect_with_retry(AGENT, AGENT_PWD, CFG["DSN"])
    try:
        for _ in range(500):
            execute_sql(c, "UPDATE race_counter SET n = n + 1 WHERE id = 1")
    finally:
        c.close()
ts = [threading.Thread(target=db_inc) for _ in range(2)]
[t.start() for t in ts]; [t.join() for t in ts]
db_final = fetch_rows(conn, "SELECT n FROM race_counter WHERE id = 1")[0]["N"]

print(f"OS file counter (no lock):   {os_final}/1000   <- lost updates" if os_final < 1000
      else f"OS file counter (no lock):   {os_final}/1000")
print(f"DB counter (atomic UPDATE):  {db_final}/1000   <- exact (ACID isolation)")

> ### ✅ Key takeaway, Part 2
> The agent now has a **scratchpad with file ergonomics and database guarantees**: progressive-disclosure
> reads, write tools it can call, durability across sessions, and ACID isolation under concurrency, all inside
> one boundary. We chose the substrate before the schema on purpose.
>

> And to be fair, you *could* bolt a file-locking layer onto plain files to fix the race we just saw. But
> that road doesn't stop: locks lead to transactions, then crash recovery, then access control, then an
> index, then backups — each fix another piece of a storage engine, until you've re-implemented a database
> with fewer guarantees and more bugs. The engineering judgment is to start from the system that already
> solved all of it.
>
> Next, **Part 3** fills the *other* half of the substrate — the searchable vector tables, via **OracleVS** —
> and builds the read path that turns a question into the right few rows.

---
# Part 3: Encoding and Retrieval

> **Where we are.** Part 2 built the *file half* of the substrate, a durable, ACID scratchpad the agent reads
> by progressive disclosure. But files are searched by keyword and grep only.
>
> **What this part does.** It builds the *table half* with **OracleVS**: the **write path** (how any text
> becomes a vector and lands in a searchable table, with the embedding produced in-database) and the **read
> path** (a ladder of retrieval techniques from keyword up to cross-encoder reranking).
>
> **Why it matters.** Retrieval *is* context engineering: it is how the harness hands the model a small,
> relevant slice of a large store instead of dumping everything into the window. Every later layer, memory, the
> semantic catalog, the toolbox, the skillbox, is read through this ladder.

### What we build in Part 3, and where it all lives

This part builds **one write path** and a **four-rung read path**, and every byte of it (text, vectors,
indexes, search) lives **inside the Oracle AI Database**. The retrieval techniques we implement, in order of
increasing power:

| # | Rung | What it is | Built with |
|---|---|---|---|
| 1 | **Keyword** | exact-term / identifier match (Oracle Text `CONTAINS`) | `langchain_oracledb` `OracleTextSearchRetriever` |
| 2 | **Vector** | meaning / paraphrase match (cosine over HNSW) | `OracleVS.similarity_search_with_score` |
| 3 | **Hybrid** | fuse keyword + vector with Reciprocal Rank Fusion | our small `reciprocal_rank_fusion` helper |
| 4 | **Rerank** | re-score the shortlist with a cross-encoder | in-database ONNX reranker (Part 1) |

And the data types this one store holds, all in one `OracleVS` table with a `namespace` to separate them:

| Data type | Example | Namespace |
|---|---|---|
| short **facts** | "Supplier concentration is the top operational risk." | `knowledge` |
| **document chunks** | overlapping windows of a longer memo | `knowledge` |
| **schema facts** (Part 5) | "ORDER_ITEMS.DISCOUNT is a 0-100 percentage" | `semantic` |

```mermaid
flowchart TB
    subgraph ORA["Oracle AI Database (one boundary)"]
        direction TB
        TXT["Text: facts, chunks, schema facts"] -->|"VECTOR_EMBEDDING (in-DB ONNX)"| TBL[("OracleVS table<br/>ID · TEXT · METADATA · EMBEDDING")]
        TBL --> HNSW[("HNSW vector index")]
        TBL --> CTX[("Oracle Text index")]
        Q["Query"] --> KW["1. keyword (CONTAINS)"] --> CTX
        Q --> VS["2. vector (cosine)"] --> HNSW
        KW --> RRF["3. hybrid: RRF fuse"]
        VS --> RRF
        RRF --> RR["4. rerank (cross-encoder)"]
        RR --> TOPK["top-k rows for the prompt"]
    end
```

Nothing leaves the database to be embedded, indexed, searched, or reranked.


### 3.1 Why OracleVS, and one store with many namespaces

The **`langchain-oracledb`** package ships **`OracleVS`**, a maintained vector store for Oracle, and **`OracleEmbeddings`**, which can compute embeddings **inside the database**.

Pairing them keeps our own code small and keeps the lesson on the ideas rather than on plumbing.

Two design points:

- **In-database embeddings are preserved.** We construct `OracleEmbeddings(provider="database", model=...)`,
  so OracleVS computes vectors with the loaded ONNX model, in the engine. No text leaves the database to be
  vectorised. The thesis from Part 0 still holds.

- **One physical table, many namespaces.** Rather than a table per concept, we use **one** OracleVS table and
  put a `namespace` in each row's metadata (`knowledge`, `semantic`, `summary`, and so on). One HNSW index and
  one text index then serve every kind of record, and a metadata filter keeps them logically separate.

OracleVS creates the table with four columns: `ID` (a RAW key), `TEXT` (the content), `METADATA` (JSON), and
`EMBEDDING` (a `VECTOR`). We will build our retrieval ladder over exactly those columns.

> 📎 **Reference.** Every `langchain_oracledb` method this part uses (the store, the embeddings, the index
> helpers, the keyword and hybrid retrievers) is documented in Oracle's official
> [LangChain OracleDB integration guide](https://docs.oracle.com/en/database/oracle/oracle-database/26/aintg/langchain-oracledb-integration-guide/langchain-python.html).


> 💡 **Key term: HNSW.** *Hierarchical Navigable Small World* is a graph index for approximate
> nearest-neighbour search. 
> 
> It trades a little recall for a large speedup versus comparing the query to every
> row. 
>
> In Oracle you select it with `ORGANIZATION INMEMORY NEIGHBOR GRAPH`. `WITH TARGET ACCURACY 95` asks
> Oracle to tune the graph to about 95% recall.

### 3.2 Create the store, with in-database embeddings

We build the embeddings object first (it runs `VECTOR_EMBEDDING` over our connection), then hand it to
`OracleVS`. From here, `store.add_texts(...)` is our entire write path.

### 🧩 TODO 5 — In-database embeddings for the vector store

The implementation in the next cell has been removed. Open **[`docs/todo5.md`](docs/todo5.md)** for a guided explanation and the solution snippet, then replace the placeholder.

Run the **`✅ TODO 5 check`** cell right after — it must pass before you continue.

In [ ]:
from langchain_oracledb.embeddings import OracleEmbeddings

# 🧩 TODO 5 — which provider makes OracleEmbeddings compute vectors INSIDE the database
# (with the ONNX model you loaded in Part 1), instead of calling an external embedding service?
# Fill in the provider value. See docs/todo5.md
db_embeddings = OracleEmbeddings(
    conn=conn,
    params={"provider": "", "model": CFG["EMBED_MODEL"]})
print("Embedder configured — fill in the provider, then run the check.")


In [ ]:
# ✅ TODO 5 check — do not edit. This must pass before you continue.
assert db_embeddings.params.get("provider") == "database", \
    "set params['provider'] = 'database' so the ONNX model embeds INSIDE Oracle (no external service)"
_v = db_embeddings.embed_query("probe")
assert isinstance(_v, list) and len(_v) == CFG["VECTOR_DIM"], \
    f"expected a {CFG['VECTOR_DIM']}-dim vector; got {len(_v) if hasattr(_v, '__len__') else type(_v)}"
print(f"✓ TODO 5 complete — text embeds to {len(_v)} dims, entirely in-database.")


### 🧩 TODO 6 — Choose the vector distance strategy

The implementation in the next cell has been removed. Open **[`docs/todo6.md`](docs/todo6.md)** for a guided explanation and the solution snippet, then replace the placeholder.

Run the **`✅ TODO 6 check`** cell right after — it must pass before you continue.

In [ ]:
from langchain_oracledb.vectorstores import OracleVS, DistanceStrategy

# 🧩 TODO 6 — pick the distance strategy the store uses to measure vector similarity.
# Options on DistanceStrategy: COSINE, EUCLIDEAN_DISTANCE, DOT_PRODUCT, MAX_INNER_PRODUCT, JACCARD.
# Which compares meaning by ANGLE (the standard for normalized text embeddings)? See docs/todo6.md
VSTORE_TABLE = "AGENT_VSTORE"
store = OracleVS(
    client=conn,
    embedding_function=db_embeddings,
    table_name=VSTORE_TABLE,
    distance_strategy=None)
print("OracleVS ready over", VSTORE_TABLE, "| distance:", getattr(store, "distance_strategy", None))


In [ ]:
# ✅ TODO 6 check — do not edit. This must pass before you continue.
from langchain_oracledb.vectorstores import DistanceStrategy
assert store.distance_strategy == DistanceStrategy.COSINE, \
    "use DistanceStrategy.COSINE — normalized text embeddings are compared by ANGLE, not raw magnitude"
print("✓ TODO 6 complete — the store ranks neighbours by COSINE distance.")


### 3.3 The write path: short facts and a chunked document

The `langchain_oracledb` methods we use from here through the end of Part 3, and what each does, so the library
calls below are not a mystery:

| Method | What it does |
|---|---|
| `OracleEmbeddings(provider="database", model=...)` | Computes embeddings **inside** the database with the loaded ONNX model (no external service). |
| `OracleVS(...).add_texts(texts, metadatas)` | Embeds and inserts rows in one call; this is our entire write path. |
| `oraclevs.create_index(conn, store, params)` | Builds the **HNSW** vector index for the store (used in 3.4). |
| `create_text_index(conn, idx_name, vector_store)` | Builds the **Oracle Text** keyword index (used in 3.4). |
| `store.similarity_search_with_score(query, k, filter)` | Vector search by meaning, with a metadata filter (the **vector** rung). |
| `OracleTextSearchRetriever(...).invoke(query)` | Oracle Text keyword search (the **keyword** rung). |

We lean on these instead of writing our own SQL, so our code stays small and the lesson stays on the ideas.

`add_texts(...)` embeds short records (one row each). For a long document we first split it into overlapping
windows, because a multi-paragraph memo is not one "meaning," it is many, and a single vector would blur them
together. We use a tiny word-window splitter so retrieval can return the *paragraph* that answers a question
rather than the whole file.

> ⚠️ A small but real gotcha: OracleVS tags each metadata dict with an internal id as it writes, so pass a
> **distinct dict per row** (a list comprehension), never `[{...}] * n`, which would alias one dict across
> every row.


First, a tiny **word-window splitter**. It cuts a long text into overlapping chunks of about `size` words, so each chunk holds roughly one idea, and the overlap keeps a sentence that straddles a boundary from being lost.

### 🧩 TODO 7 — Chunk text into overlapping windows

The implementation in the next cell has been removed. Open **[`docs/todo7.md`](docs/todo7.md)** for a guided explanation and the solution snippet, then replace the placeholder.

Run the **`✅ TODO 7 check`** cell right after — it must pass before you continue.

In [ ]:
def chunk_words(text, size=200, overlap=20):
    """Split text into overlapping word windows (one idea per chunk)."""
    # 🧩 TODO 7 — split text into overlapping word windows for embedding. See docs/todo7.md
    raise NotImplementedError("Complete TODO 7 — open docs/todo7.md")


In [ ]:
# ✅ TODO 7 check — do not edit.
assert chunk_words("just a few words") == ["just a few words"], "text shorter than `size` returns one chunk unchanged"
_words = " ".join(str(i) for i in range(450))
_chunks = chunk_words(_words, size=200, overlap=20)
assert len(_chunks) >= 2, "450 words at size=200 should split into multiple chunks"
assert all(len(c.split()) <= 200 for c in _chunks), "no chunk may exceed `size` words"
assert _chunks[0].split()[-20:] == _chunks[1].split()[:20], "consecutive chunks must overlap by `overlap` words"
print("✓ TODO 7 complete — text is split into overlapping windows ready for embedding.")


**The short facts.** The first write call stores a handful of standalone **facts**, one row each. A "fact" here
is a single declarative knowledge statement (the kind of line that would sit in a `NOTES.md`), short enough
that the whole sentence is one coherent meaning, so it embeds cleanly into a single vector. `add_texts` embeds
and inserts them in one call.


In [ ]:
facts = ["The Outdoors category drove Q3 revenue growth.",
         "Supplier concentration is the top operational risk this quarter.",
         "A dual-sourcing decision for Outdoors is targeted for Q1.",
         "Customer churn rises sharply when delivery exceeds five days.",
         "Returns spike in the Apparel category right after the holidays."]
store.add_texts(facts, metadatas=[{"namespace": "knowledge", "src": "q3"} for _ in facts])

**The longer memo.** Next we write a multi-paragraph operations review. Unlike a one-line fact, a **memo**
covers several distinct ideas, so embedding it whole would blur them into one muddy vector. We split it into
overlapping word windows with `chunk_words` first, so each chunk is about one idea and retrieval can return the
exact paragraph that answers a question rather than the whole document.


In [ ]:
memo = ("Operations review. Fulfilment lead time is the single biggest driver of customer churn: when "
        "delivery slips past five days, repeat-purchase rates fall by roughly a third. The other standing "
        "risk is supplier concentration in Outdoors, where one vendor covers about forty percent of cost of "
        "goods sold; qualifying a second source is the agreed mitigation for the first quarter.")
chunks = chunk_words(memo, size=60, overlap=10)

In [ ]:
store.add_texts(chunks, metadatas=[{"namespace": "knowledge", "src": "ops_memo"} for _ in chunks])

n_know = fetch_rows(conn, f"SELECT COUNT(*) n FROM {VSTORE_TABLE} WHERE JSON_VALUE(metadata,'$.namespace')='knowledge'")[0]["N"]

print("knowledge rows:", n_know, "(all embeddings produced in-database by OracleVS)")

### 3.4 Add the two indexes the ladder needs

OracleVS created the table; now we add the two indexes our read path uses: an **HNSW** vector index on
`EMBEDDING` for search by meaning, and an **Oracle Text** (`CONTEXT`) index on `TEXT` for exact keyword search.
These are the two halves of hybrid retrieval.

> 💡 **A note on the SQL, for the SQL-averse.** This notebook is database-native, so you will see DDL
> (`CREATE TABLE`, `CREATE ... INDEX`). You do not have to author it from scratch. Two things shrink it: first,
> **OracleVS already created the vector table for us** with no DDL on our part; second, the one piece of DDL we
> repeat, the vector index, we wrap **once** in a friendly Python helper, `create_hnsw_index(table)`, and call
> that everywhere after. So in practice you write the scary index SQL exactly zero more times. We define the
> helpers here and reuse them for every store in Parts 4, 6, and 8.

In [ ]:
from langchain_oracledb.vectorstores import oraclevs
from langchain_oracledb.retrievers.text_search import create_text_index as lc_create_text_index

# AGENT_VSTORE is an OracleVS store, so we let langchain_oracledb build its indexes for us.
#   - oraclevs.create_index: the HNSW vector index (search by meaning).
#   - create_text_index:     the Oracle Text keyword index (exact-term search).
oraclevs.drop_index_if_exists(conn, "AGENT_VSTORE_HNSW")

oraclevs.create_index(conn, store, params={"idx_name": "AGENT_VSTORE_HNSW", "idx_type": "HNSW",
                                            "accuracy": 95, "parallel": 4})

try:
    lc_create_text_index(conn, idx_name="AGENT_VSTORE_TEXT", vector_store=store)
except Exception as e:
    if "ORA-00955" not in str(e):   # ORA-00955 = name already in use -> safe to ignore on re-run
        raise

In [ ]:

# The registry tables in Parts 4, 6, and 8 are plain (non-OracleVS) tables we create ourselves, so the
# store-bound helper above does not apply to them. We keep one tiny DDL helper for their vector indexes.

def create_hnsw_index(table, column="embedding", accuracy=95, name=None):
    name = name or f"{table.lower()}_hnsw"
    ddl_idempotent(conn, f'''CREATE VECTOR INDEX {name} ON {table} ({column})
      ORGANIZATION INMEMORY NEIGHBOR GRAPH DISTANCE COSINE WITH TARGET ACCURACY {accuracy}''')

idx = fetch_rows(conn, f"SELECT index_name, index_type FROM user_indexes WHERE table_name='{VSTORE_TABLE}' ORDER BY 1")

print("AGENT_VSTORE indexes (built by langchain_oracledb):", [(i["INDEX_NAME"], i["INDEX_TYPE"]) for i in idx])
print("create_hnsw_index helper ready for the registry tables.")


### 3.5 The read path: a retrieval ladder

A query must return the *right small set* of rows, not everything related. We build a four-rung ladder,
each rung more powerful (and more expensive) than the last: **keyword** (Oracle Text via
`OracleTextSearchRetriever`), **vector** (cosine via `similarity_search_with_score`), **hybrid** (fuse the
two with Reciprocal Rank Fusion), and **rerank** (the in-database cross-encoder). We lean on
`langchain_oracledb` retrievers where they exist and write only the thin glue ourselves; each rung returns
a common dict-row shape so the next can fuse and rerank it. We start with the first two rungs.

### 🧩 TODO 8 — Semantic (vector) retrieval

The implementation in the next cell has been removed. Open **[`docs/todo8.md`](docs/todo8.md)** for a guided explanation and the solution snippet, then replace the placeholder.

Run the **`✅ TODO 8 check`** cell right after — it must pass before you continue.

In [ ]:
from langchain_oracledb.retrievers.text_search import OracleTextSearchRetriever

# Keyword retriever: Oracle Text CONTAINS search, provided by langchain_oracledb (no hand-written SQL).
# Built once over the AGENT_VSTORE text index and reused by the ladder. We pull a wide pool (k=30) and
# narrow by namespace in Python, so one retriever serves every namespace.
_keyword_retriever = OracleTextSearchRetriever(vector_store=store, k=30, fuzzy=True, return_scores=True)

def _as_row(doc, score):
    """Adapt a langchain Document to the dict-row shape the rest of the ladder already uses."""
    return {"ID": doc.id, "TEXT": doc.page_content, "META": doc.metadata, "score": score}

def keyword_search(query, namespace=None, k=5):
    docs = _keyword_retriever.invoke(query)
    if namespace:
        docs = [d for d in docs if d.metadata.get("namespace") == namespace]
    return [_as_row(d, d.metadata.get("score")) for d in docs[:k]]

def vector_search(query, namespace=None, k=5):
    # 🧩 TODO 8 — semantic retrieval: rank the store by cosine distance to the query. See docs/todo8.md
    raise NotImplementedError("Complete TODO 8 — open docs/todo8.md")

print("Rungs 1-2 ready: keyword_search (OracleTextSearchRetriever) + vector_search (similarity_search_with_score).")


In [ ]:
# ✅ TODO 8 check — do not edit.
_hits = vector_search("revenue growth", k=3)
assert isinstance(_hits, list) and _hits, "vector_search should return ranked rows from the store"
assert {"ID", "TEXT", "META", "score"} <= set(_hits[0]), "each row carries ID / TEXT / META / score"
print("✓ TODO 8 complete — semantic retrieval works. Top hit:", str(_hits[0]["TEXT"])[:60])


### 3.6 Hybrid search, fused with Reciprocal Rank Fusion

Keyword and vector each miss what the other catches (exact terms vs paraphrase), so hybrid search runs both
and merges the ranked lists. The clean, weight-free merge is **Reciprocal Rank Fusion (RRF)**: each row
earns `1/(c + rank)` from every list it appears in, and the sums decide the order — a row near the top of
*both* lists beats one that is first in only one. We pull the fusion into its own small helper so the
technique is easy to read, test, and reuse.

### 🧩 TODO 9 — Fuse keyword + vector results with Reciprocal Rank Fusion

The implementation in the next cell has been removed. Open **[`docs/todo9.md`](docs/todo9.md)** for a guided explanation and the solution snippet, then replace the placeholder.

Run the **`✅ TODO 9 check`** cell right after — it must pass before you continue.

In [ ]:
def reciprocal_rank_fusion(ranked_lists, c=60):
    """Merge several ranked lists into one. Each row earns 1/(c+rank) from every list it appears in, so rows
    found by more than one method add up and rise. Returns rows sorted best-first, each tagged with 'rrf'."""
    # 🧩 TODO 9 — fuse the ranked lists with reciprocal rank fusion. See docs/todo9.md
    raise NotImplementedError("Complete TODO 9 — open docs/todo9.md")

def hybrid_search(query, namespace=None, k=5, pool=20):
    """Run keyword + vector over a candidate pool, then fuse with RRF and keep the top k."""
    vector_hits = vector_search(query, namespace, pool)
    keyword_hits = keyword_search(query, namespace, pool)
    return reciprocal_rank_fusion([vector_hits, keyword_hits])[:k]

print("Rung 4 ready: hybrid_search, built on a reusable reciprocal_rank_fusion helper.")


In [ ]:
# ✅ TODO 9 check — do not edit.
_l1 = [{"ID": "a"}, {"ID": "b"}, {"ID": "c"}]   # e.g. the vector hits
_l2 = [{"ID": "b"}, {"ID": "d"}]                # e.g. the keyword hits
_fused = reciprocal_rank_fusion([_l1, _l2])
assert isinstance(_fused, list) and _fused, "reciprocal_rank_fusion should return a non-empty list"
assert {r["ID"] for r in _fused} == {"a", "b", "c", "d"}, "every row from every list must survive the fusion"
assert _fused[0]["ID"] == "b", "'b' is found by BOTH lists near the top, so RRF must rank it first"
assert "rrf" in _fused[0], "each fused row should be tagged with its 'rrf' score"
print("✓ TODO 9 complete — keyword + vector results fuse into one ranked list. Top:", _fused[0]["ID"])


### 3.7 Rerank: re-score the shortlist with the cross-encoder

`rerank` re-scores a shortlist with the in-database cross-encoder, which reads the query and each document
*together* — `PREDICTION(model USING (:q || ' [SEP] ' || doc) AS DATA)`. It returns one raw relevance
number per pair (higher = more relevant); we keep it raw because what matters is the **ordering**, not the
absolute value. If the reranker is not loaded, this rung transparently returns the input order.

### 🧩 TODO 10 — Rerank the shortlist with a cross-encoder

The implementation in the next cell has been removed. Open **[`docs/todo10.md`](docs/todo10.md)** for a guided explanation and the solution snippet, then replace the placeholder.

Run the **`✅ TODO 10 check`** cell right after — it must pass before you continue.

In [ ]:
def rerank(query, candidates, k=5):
    """Cross-encoder rescoring via the in-DB reranker. Tags each row with 'rerank_score' (higher = more
    relevant) and returns them reordered by it, or the input order if the reranker is unavailable."""
    # 🧩 TODO 10 — rescore with the in-DB cross-encoder; fall back to input order if it is unavailable. See docs/todo10.md
    raise NotImplementedError("Complete TODO 10 — open docs/todo10.md")

print("Rung 5 ready: rerank (raw cross-encoder score; higher = more relevant).")


In [ ]:
# ✅ TODO 10 check — do not edit.
_cands = [{"ID": 1, "TEXT": "supplier risk concentrated in the Outdoors category"},
          {"ID": 2, "TEXT": "an unrelated note about shipping-label printers"}]
_r = rerank("main operational risk", _cands, k=5)
assert isinstance(_r, list) and len(_r) == 2, "rerank returns the (reordered) candidates, capped at k"
assert all("rerank_score" in c for c in _r), "every returned row must be tagged with a 'rerank_score'"
print("✓ TODO 10 complete — the rerank rung is wired (cross-encoder when loaded; graceful fallback otherwise).")


### 3.8 Checkpoint: see the ladder, and watch reranking reorder

First, the same question answered by keyword, vector, and the OracleVS one-liner, so you can see them agree on
the obvious hit. Then the part worth studying: a **side-by-side table** showing the hybrid (vector-fusion)
ranking versus the reranked ranking, with each document's **position movement**. This is where the
cross-encoder earns its keep: it reads query and document together and pushes the genuinely-relevant rows up,
even when the bi-encoder left them mid-pack. The `score` column is the raw cross-encoder relevance (higher is
better); read the **movement**, not the absolute number.

In [ ]:
import pandas as pd

Qy = "What is the main operational risk?"

# keyword and vector are now both langchain_oracledb retrievers; they agree on the obvious top hit.
print("[keyword] ", [str(r["TEXT"])[:55] for r in keyword_search(Qy, "knowledge", 3)])
print("[vector]  ", [str(r["TEXT"])[:55] for r in vector_search(Qy, "knowledge", 3)])


In [ ]:
# Position movement: hybrid order -> reranked order, side by side.
shortlist = hybrid_search(Qy, "knowledge", k=6)
before = [str(r["TEXT"]) for r in shortlist]
after = rerank(Qy, shortlist, k=6)
after_text = [str(r["TEXT"]) for r in after]

In [ ]:

rows = []
for new_pos, r in enumerate(after, 1):
    txt = str(r["TEXT"]); old_pos = before.index(txt) + 1
    delta = old_pos - new_pos
    move = "same" if delta == 0 else (f"up {delta}" if delta > 0 else f"down {-delta}")
    rows.append({"rerank_pos": new_pos, "vector_pos": old_pos, "move": move,
                 "score": None if r["rerank_score"] is None else round(r["rerank_score"], 2),
                 "document": txt[:60]})
df = pd.DataFrame(rows, columns=["rerank_pos", "vector_pos", "move", "score", "document"])
df

In [ ]:
print("\nReranking reorders the shortlist (vector_pos -> rerank_pos):")
print(df.to_string(index=False))
print("\nPART 3 OK.   RERANK_AVAILABLE =", RERANK_AVAILABLE)

> ### ✅ Key takeaway, Part 3
> Both halves of the substrate now exist: a **file store** (Part 2) read by grep and slice, and a **vector
> store** built on **OracleVS** read by a **retrieval ladder** (keyword, vector, hybrid, rerank). OracleVS gave
> us the write path and a one-line vector search for almost no code, while the ladder we layered on top stays
> fully in our control. Crucially, retrieval is *progressive disclosure for tables*: hand the model the few
> rows it needs, nothing more. With storage and retrieval solved, **Part 4** adds the *cognitive* layer: a
> memory taxonomy backed by the Oracle AI Agent Memory package, fact extraction, and the bridge that promotes
> scratch files into long-term memory.

---
# Part 4: Cognitive Memory (the Oracle AI Agent Memory package)

> **Where we are.** We have two storage media (files plus vector tables) and a retrieval ladder. That is
> *mechanics*. It is not yet *memory* in the cognitive sense. There is no notion of "what happened," "what I
> know about this user," or "what I am working on right now."
>
> **What this part does.** It adds the cognitive layer using a real memory system of record, the **Oracle AI
> Agent Memory** package (OAMP), wired to our **in-database embedder**. It turns on OAMP's **fact extraction**,
> so raw text is distilled into clean atomic memories. It then builds the **promotion pipeline** that moves
> notes from the scratch filesystem into long-term memory on a schedule.
>
> **Why it matters.** This is the heart of *memory engineering*: giving the agent a persistent identity whose
> knowledge survives process death and accumulates over time. An agent without this is an amnesiac that
> re-learns everything every session.

![What OAMP absorbs — one line (mem.context_card) instead of a whole memory subsystem](images/what_oamp_absorbs.png)

### The memory taxonomy

Human memory is not one thing, and neither is agent memory. Naming the types tells you *where each kind of
knowledge should live* and *how it should be written and read*.

| Human memory | Agent memory | What it holds | Backed by |
|---|---|---|---|
| **Episodic** | conversation history | what was said, turn by turn, per thread | OAMP threads |
| **Semantic** | facts and knowledge | durable truths about the user, the domain, the world | OAMP memories (extracted) |
| **Working** | the scratchpad of the mind | a running summary plus a context card for the live task | OAMP thread summary / card |
| **Procedural** | how-to | tools, reusable workflow recipes, skills | `agent_workflow` table plus the skillbox (Part 6) |

> ⚠️ **Add one memory type at a time.** Reaching for elaborate multi-agent shared memory when a single
> conversation store would do is the agent equivalent of premature optimization. We add episodic and semantic
> first, working and procedural only when a behaviour demands them, which is the order this notebook introduces
> them.

### 4.1 The in-database embedder, and initialising OAMP

OAMP accepts a custom **embedder**. We give it one whose `embed()` runs `VECTOR_EMBEDDING` over the same
connection, so OAMP stores the *same* 384-dim vectors as everything else, computed in-engine, with no
external embedding service.

> 💰 One SQL round-trip per text — fine at this notebook's volumes; batch it at production scale.

### 🧩 TODO 11 — Embed text inside the database

The implementation in the next cell has been removed. Open **[`docs/todo11.md`](docs/todo11.md)** for a guided explanation and the solution snippet, then replace the placeholder.

Run the **`✅ TODO 11 check`** cell right after — it must pass before you continue.

In [ ]:
import numpy as np
from oracleagentmemory.core import OracleAgentMemory
from oracleagentmemory.core.llms import Llm
from oracleagentmemory.apis.embedders.embedder import IEmbedder

class InDBOnnxEmbedder(IEmbedder):
    """An OAMP embedder that computes vectors with the in-database ONNX model (no external service)."""
    def __init__(self, conn, model):
        self.conn = conn; self.model = model
    def embed(self, texts, *, is_query=False):
        # 🧩 TODO 11 — embed each text to a 384-dim vector INSIDE Oracle via VECTOR_EMBEDDING. See docs/todo11.md
        raise NotImplementedError("Complete TODO 11 — open docs/todo11.md")
    async def embed_async(self, texts, *, is_query=False):
        return self.embed(texts, is_query=is_query)

print("InDBOnnxEmbedder ready.")


In [ ]:
# ✅ TODO 11 check — do not edit.
_emb = InDBOnnxEmbedder(conn, CFG["EMBED_MODEL"])
_v = _emb.embed(["smoke test"])
assert hasattr(_v, "shape") and tuple(_v.shape) == (1, 384), \
    f"embed(['..']) should return a (1, 384) array; got shape {getattr(_v, 'shape', None)}"
_v3 = _emb.embed(["one", "two", "three"])
assert tuple(_v3.shape) == (3, 384), "one 384-dim row per input text"
print("✓ TODO 11 complete — text embedded to 384 dims, entirely inside Oracle.")


### 4.2 Turning on fact extraction, and what it does

We initialise OAMP with `extract_memories=True`. Here is what that flag does, because it is easy to picture the
wrong way.

Extraction works on the **conversation path**. 

When you add dialogue turns to a thread, OAMP reads the conversation, pulls out the **atomic, durable facts** worth remembering, and writes those clean statements into
semantic memory automatically (skipping the chit-chat that is not worth keeping). 

So a rambling exchange that includes *"by the way I think we probably want refunds above 500 dollars to need a manager to sign off, and I
am based in Berlin"* yields crisp memories such as *"Refunds over $500 require manager approval."* and *"The
user is based in Berlin."* The payoff is that later recall returns sharp facts rather than raw transcript,
which is exactly what you want to drop into a prompt. This is also why it matters in the agent loop (Part 7):
as the agent converses, it *learns durable facts from the dialogue without anyone calling a save function*.

A direct `add_memory("...")` is the other path: it stores the text **verbatim**, which is what you want when
you have an explicit fact to record exactly. Extraction is for distilling *conversations*; `add_memory` is for
saving *facts*. We use both.

> 💰 **The trade-off, named.** Extraction costs an **LLM call when conversation turns are processed**. That is
> the right price for learning from dialogue (you read the facts far more often than you extract them), which
> is why we enable it. You would not enable it on a high-volume non-conversational firehose.

Here is what each argument to `OracleAgentMemory(...)` in the next cell does:

| Argument | What it does |
|---|---|
| `connection` | The `oracledb` connection OAMP uses for its own tables. We pass the same `AGENT` connection, so memory lives in the same schema as everything else. |
| `embedder` | How OAMP turns text into vectors. We give it our in-database ONNX embedder, so memory shares the one 384-dim vector space and never calls an external embedding service. |
| `llm` | The model OAMP uses for its *own* LLM work: extracting facts from conversations and writing thread summaries (the context card). |
| `extract_memories` | `True` distils durable facts from conversation turns into semantic memory (this section). `False` stores only what you explicitly add. |
| `table_name_prefix` | The prefix for the tables OAMP creates (`OAMP_`), so they are easy to spot and never collide with ours. |
| `schema_policy` | `"create_if_necessary"` lets OAMP create its tables the first time it runs, then reuse them on later runs. |

In [ ]:
USER_ID, AGENT_ID = "analyst", "total_recall"

def make_oamp_llm():
    """OAMP's extraction/summary LLM -- the same OCI GenAI (OpenAI-compatible) endpoint as the chat
    model, wired through OAMP's LiteLLM layer as openai/<model>."""
    if CFG["LLM_PROVIDER"] == "oci":
        return make_rotating_oamp_llm()
    return Llm(f"openai/{CFG['LLM_MODEL']}")

oamp = OracleAgentMemory(
    connection=conn,
    embedder=InDBOnnxEmbedder(conn, CFG["EMBED_MODEL"]),
    llm=make_oamp_llm(),
    extract_memories=CFG["OAMP_EXTRACT"],          # True: distil facts from conversation turns (not direct add_memory)
    table_name_prefix=CFG["OAMP_PREFIX"],
    schema_policy="create_if_necessary",
)

In [ ]:
# add_user/add_agent raise if the actor already exists, so guard them for re-runnability.
for _fn, _args in [(oamp.add_user, (USER_ID, "An analytics engineer who prefers concise, decision-first answers.")),
                   (oamp.add_agent, (AGENT_ID, "A retail-analytics agent that grounds answers in the database and automates them."))]:
    try:
        _fn(*_args)
    except Exception:
        pass  # actor already exists
print("OAMP ready (extract_memories=%s). Tables:" % CFG["OAMP_EXTRACT"],
      [t["TABLE_NAME"] for t in fetch_rows(conn, "SELECT table_name FROM user_tables WHERE table_name LIKE 'OAMP%' ORDER BY 1")])

### 4.2b See extraction in action

We hold a short, deliberately chatty conversation in a thread, then look at what landed in durable memory.
Nobody called a save function: the crisp, standalone statements in the output (for example *"Any refund over
$500 should require manager approval."* and *"User is based in Berlin."*) were **extracted from the dialogue**
by OAMP because extraction is on. (We use the thread API directly here to keep the demo self-contained; from
Part 4.3b onward the same thing happens through `MemoryManager`.)

> 💡 A search returns **both** the distilled facts *and* the original turns they came from (the raw messages
> are stored and searchable too). So among the results you will see the chatty sentence next to the clean fact
> it produced. The point of extraction is the short, declarative statements; those are what later recall feeds
> into a prompt.

> 💡 Extraction is **asynchronous**, so we `await add_messages_async(...)` here to let it finish before we
> read. In the live agent loop the ordinary synchronous `add_turn` is fine, because extraction completes in the
> background between turns; we only need to await it here so the very next line can show the result.

In [ ]:
# Idempotent: a previous run (or the shared schema) may already hold this demo thread, and
# create_thread() raises "Thread already exists". Drop it first so re-running starts clean.
try:
    oamp.delete_thread("extract_demo")
except Exception:
    pass
extract_thread = oamp.create_thread(thread_id="extract_demo", user_id=USER_ID, agent_id=AGENT_ID)

await extract_thread.add_messages_async([
    {"role": "user", "content": "So anyway, thinking out loud, but we should make any refund over $500 "
                                "need a manager to approve it. Also I'm based in Berlin and prefer metric units."},
    {"role": "assistant", "content": "Understood, I will keep that in mind."},
])
print("durable facts AUTO-EXTRACTED from the conversation (no add_memory call):")
for fact in [r.content for r in oamp.search("refund policy and user preferences",
                                            user_id=USER_ID, agent_id=AGENT_ID, max_results=5)]:
    print("  -", fact)

### 4.3 The procedural store: workflow recipes that record their outcome

Before the facade, we create the **procedural** store. A *workflow recipe* is a thing the agent did once,
captured so it can be reused. Two design decisions matter here:

- **Outcome tracking.** Each capture records whether the run **succeeded or failed**, and we keep running
  `successes` and `failures` counts. This is what lets the harvester later promote only the workflows that
  *actually work*, and lets the loop avoid re-suggesting a path that keeps failing.
  
- **A `promoted` flag.** Once a recurring recipe is distilled into a skill (Part 6), we set `promoted='Y'`.
  From then on we **stop retrieving it into the agent's context**, because the skill is now the canonical,
  better-written version. Retrieving both would waste context and risk contradicting guidance.

> 📎 **Why a plain table here, and not OAMP records?** OAMP can store custom records (through its store API,
> with an auto-computed embedding and JSON metadata), so it is fair to ask why we do not drop this table and let
> OAMP own it too. We keep it deliberately, for one concrete reason: the harvester (Part 6) promotes a recipe
> only when `successes > failures`, and it increments those counters on every run. OAMP keeps such numbers
> inside a JSON `metadata` blob, which supports equality filters but **not** SQL arithmetic (`WHERE successes >
> failures`) or atomic counter increments. A real `NUMBER` column gives us both for free. Rule of thumb: store
> *facts* in OAMP (Part 4), but keep a typed table when you need server-side numeric logic, which is exactly
> the procedural store's job.


In [ ]:
create_table(conn, '''CREATE TABLE agent_workflow (
  id RAW(16) DEFAULT SYS_GUID() PRIMARY KEY, intent VARCHAR2(400), steps CLOB,
  tools_used VARCHAR2(1000), occurrences NUMBER DEFAULT 1,
  successes NUMBER DEFAULT 0, failures NUMBER DEFAULT 0, last_outcome VARCHAR2(10),
  promoted CHAR(1) DEFAULT 'N',
  embedding VECTOR(384, FLOAT32), created_at TIMESTAMP DEFAULT SYSTIMESTAMP,
  last_seen TIMESTAMP DEFAULT SYSTIMESTAMP)''')   # create_table() heals schema drift (see Part 0 helpers)
create_hnsw_index("agent_workflow")   # same friendly helper from Part 3
print("agent_workflow ready (successes / failures / last_outcome / promoted; schema auto-reconciled).")

### 4.3b One facade over all memory

The agent should not care whether a piece of knowledge is an OAMP memory, an OAMP thread message, or a row in
the workflow table. `MemoryManager` is the single interface: **reads return LLM-ready text; writes persist.**
It delegates semantic and episodic memory to OAMP and keeps **procedural workflow memory** in our dedicated
table, because recognising a *recurring* workflow needs embedding-similarity dedup, which is our own logic and
the seed of the skill harvester in Part 6.

Read the procedural methods closely:

- `capture_workflow(intent, steps, tools, success)` dedups by meaning, bumps `occurrences`, and records the
  outcome into `successes` or `failures`.
- `recall_workflow(query)` returns the closest recipes **but only un-promoted ones**, so promoted recipes drop
  out of retrieval in favour of their skill.

In [ ]:
class MemoryManager:
    def __init__(self, oamp, conn, user_id, agent_id):
        self.oamp = oamp; self.conn = conn; self.user_id = user_id; self.agent_id = agent_id
        self._threads = {}

    # --- semantic memory: remember() stores an explicit fact verbatim; conversation turns (add_turn,
    #     below) are what OAMP extracts facts FROM when extract_memories=True ---
    def remember(self, content, metadata=None):
        return self.oamp.add_memory(content, user_id=self.user_id, agent_id=self.agent_id)
    def recall(self, query, k=5):
        res = self.oamp.search(query, user_id=self.user_id, agent_id=self.agent_id, max_results=k)
        return [r.content for r in res]

    # --- episodic memory (conversation threads) ---
    def _thread(self, thread_id):
        if thread_id not in self._threads:
            try:
                self._threads[thread_id] = self.oamp.create_thread(
                    thread_id=thread_id, user_id=self.user_id, agent_id=self.agent_id)
            except Exception:
                self._threads[thread_id] = self.oamp.get_thread(thread_id)
        return self._threads[thread_id]
    def add_turn(self, thread_id, role, content):
        self._thread(thread_id).add_messages([{"role": role, "content": content}])
    def get_turns(self, thread_id):
        try:
            return [(m.role, m.content) for m in self._thread(thread_id).get_messages()]
        except Exception:
            return []
    def thread_summary(self, thread_id):
        try:
            return self._thread(thread_id).get_summary().content
        except Exception:
            return None
    def context_card(self, thread_id, recent=6, relevant=4):
        # OAMP's prompt-assembly primitive: summary + relevant memories + recent turns, ready for the prompt.
        try:
            card = self._thread(thread_id).get_context_card(
                fallback_message_count=recent, max_relevant_results=relevant, max_recent_messages=recent)
            return card.content
        except Exception:
            return None

    # --- procedural memory (workflow capture, with similarity dedup AND outcome tracking) ---
    def capture_workflow(self, intent, steps, tools_used, success=True):
        sims = fetch_rows(self.conn, f'''SELECT id, occurrences,
                     VECTOR_DISTANCE(embedding, VECTOR_EMBEDDING({CFG['EMBED_MODEL']} USING :q AS DATA), COSINE) d
                 FROM agent_workflow ORDER BY d FETCH APPROX FIRST 1 ROWS ONLY''', {"q": intent})
        outcome = "success" if success else "failure"
        col = "successes" if success else "failures"
        if sims and sims[0]["D"] is not None and sims[0]["D"] < 0.15:
            execute_sql(self.conn, f'''UPDATE agent_workflow
                  SET occurrences=occurrences+1, {col}={col}+1, last_outcome=:o, last_seen=SYSTIMESTAMP
                  WHERE id=:i''', {"o": outcome, "i": sims[0]["ID"]})
            return sims[0]["ID"], "merged"
        execute_sql(self.conn, f'''INSERT INTO agent_workflow (intent, steps, tools_used, {col}, last_outcome, embedding)
              VALUES (:i, :s, :u, 1, :o, VECTOR_EMBEDDING({CFG['EMBED_MODEL']} USING :i AS DATA))''',
          {"i": intent, "s": json.dumps(steps), "u": ",".join(tools_used), "o": outcome})
        return None, "inserted"
    def recall_workflow(self, query, k=3, include_promoted=False):
        where = "" if include_promoted else "WHERE promoted='N'"
        return fetch_rows(self.conn, f'''SELECT id, intent, steps, tools_used, occurrences, successes, failures,
                     VECTOR_DISTANCE(embedding, VECTOR_EMBEDDING({CFG['EMBED_MODEL']} USING :q AS DATA), COSINE) d
                 FROM agent_workflow {where} ORDER BY d FETCH APPROX FIRST :k ROWS ONLY''', {"q": query, "k": k})


In [ ]:
mem = MemoryManager(oamp, conn, USER_ID, AGENT_ID)
print("MemoryManager ready (semantic+episodic via OAMP, procedural with outcomes via agent_workflow).")

### 4.4 Working memory: the OAMP context card

When you assemble a prompt for an agent, you usually need three things glued together: a **summary** of the
conversation so far, the **most relevant durable memories** for the current question, and the **last few raw
turns**. Doing that by hand means three retrievals and a lot of bespoke formatting.

The **context card** does it in one call. `thread.get_context_card(...)` returns a compact, token-bounded,
prompt-ready block:

```
<context_card>
  <topics> ... </topics>                 # what this thread is about
  <summary> ... </summary>               # synthesised rolling summary (working memory)
  <relevant_information>                 # durable memories retrieved for the current context
    <memory> ... </memory>
  </relevant_information>
  <recent_messages> ... </recent_messages>   # the last N raw turns, verbatim
</context_card>
```

It takes three knobs (`fallback_message_count`, `max_relevant_results`, `max_recent_messages`) that bound how
much of each layer is included, which is precisely the *budgeting* lever context engineering is about.

> 💰 **Not free, but cheap in the right way.** The summary inside the card is produced by the LLM, so building
> a card has a cost. The point is that it is **one bounded call per user turn** instead of re-sending an
> ever-growing transcript on every model call. In Part 7 the loop computes the card **once per user turn** and
> caches it; in Part 8 we *measure* how it keeps the window flat.

![The context card the agent actually reads, annotated part by part](images/context_card_what_an_agent_needs.png)

#### What OAMP takes off your plate (and the agent's)

The context card is not just a convenience function. It absorbs a whole category of **cognitive load** that you
would otherwise have to design, build, tune, and maintain yourself, and that the agent would otherwise have to
reason about mid-task. Working memory is deceptively hard: you have to decide *what* to summarise and *when*,
how to keep the high-signal facts while dropping filler, which durable memories are relevant to *this* turn, and
how to bound the whole thing to a token budget. OAMP owns all of that.

```mermaid
flowchart LR
    subgraph BEFORE["WITHOUT OAMP - you carry the load"]
        direction TB
        B1["Decide when to summarise"]
        B2["Write a summariser that<br/>keeps high signal, drops filler"]
        B3["Rank + retrieve relevant<br/>durable memories per turn"]
        B4["Extract durable facts from<br/>chatty conversation"]
        B5["Glue summary + memories +<br/>recent turns into a prompt"]
        B6["Bound it all to a token budget"]
    end
    subgraph AFTER["WITH OAMP - one call"]
        direction TB
        A1["mem.context_card(thread_id)"]
        A1 --> A2["bounded, prompt-ready block:<br/>topics · summary · relevant<br/>memories · recent turns"]
    end
    BEFORE -->|"OAMP absorbs this"| AFTER
```

The payoff is twofold: **the developer** writes one line instead of a memory subsystem, and **the agent** spends
its attention on the task instead of on housekeeping its own memory. That is the core promise of *memory
engineering*: make the hard, easy-to-get-wrong parts of remembering a solved, reusable component.


In [ ]:
# A few turns plus a durable fact, assembled into one prompt-ready block.
mem.add_turn("cc_demo", "user", "What drove Q3 growth?")
mem.add_turn("cc_demo", "assistant", "The Outdoors category, up 12% QoQ.")
mem.add_turn("cc_demo", "user", "What is the main risk to that?")
print(mem.context_card("cc_demo", recent=4, relevant=3) or "(context card unavailable)")

### 4.5 The promotion pipeline: scratch to long-term memory, on a schedule

> 💡 **Key term: memory promotion (a memory-engineering technique).** *Promotion* is the deliberate act of
> moving a piece of information **up the memory hierarchy**: from cheap, transient **short-term** storage (the
> scratchpad, where the agent jots freely without committing) into durable, searchable **long-term** memory.
> It matters because the two stores have opposite economics. Short-term memory is cheap to write and safe to be
> wrong in, but it is not searched by meaning and does not survive a session. Long-term memory is the opposite:
> you want only the *worth-keeping* signal in it, embedded and indexed, because everything stored there competes
> for retrieval attention later. Promotion is the valve between them, the mechanism that decides **what
> graduates**. Running it **on a schedule** rather than on every write keeps the long-term store curated without
> the agent ever stopping to do its own housekeeping.

This is the bridge that joins the two halves of the substrate, and it is **continual** by design. The scratch
filesystem (Part 2) is where the agent jots intermediate notes cheaply. Some of those notes turn out to be
worth keeping. *Promotion* moves a scratch note into durable, searchable memory, and we run it **on a schedule**
so it happens without the agent having to remember to.

We split the work along the natural seam:

- **A database-scheduled producer.** A `DBMS_SCHEDULER` job runs a stored procedure that finds un-promoted
  scratch files, extracts their text (`UTL_TO_TEXT` handles many document formats), splits them into chunks,
  and stages the chunks in a queue, all inside the database, on a cadence.
- **An OAMP consumer.** A small worker drains the queue, writing each chunk into long-term memory **through
  OAMP** (`mem.remember`), where it becomes durable and searchable. OAMP is a client library, so this step runs
  in the agent's runtime; you schedule it from your orchestrator, or, as here, call it from the harness.

We start with a reusable job helper and the staging queue.


#### The scheduled-job helper, and the role of `DBMS_SCHEDULER`

The promotion pipeline (and the catalog refresh in Part 5, and the harvester in Part 6) all need to run work
**on a cadence, inside the database**. That is what **`DBMS_SCHEDULER`** is for: it is Oracle's built-in job
scheduler, a cron-like service that lives *in* the engine and runs PL/SQL (or OS) jobs on a repeat interval,
with no external scheduler, daemon, or cron process to operate. A job keeps running even if every client
disconnects, which is exactly what you want for background maintenance.

`create_or_replace_job(name, plsql_action, repeat_interval)` wraps the two calls you make for every job,
`DBMS_SCHEDULER.DROP_JOB` (so re-running is safe) then `DBMS_SCHEDULER.CREATE_JOB`, with:

- `job_type => 'PLSQL_BLOCK'` and `job_action` = the PL/SQL to run (e.g. `BEGIN stage_scratch_for_promotion; END;`),
- `repeat_interval` = a calendar string like `FREQ=HOURLY;INTERVAL=1`,
- `enabled => TRUE` so it starts immediately.

We define it once and reuse it for the promotion producer, the semantic-catalog refresh, and the skill
harvester, so "do this in the database, on a schedule" is a one-liner every time.


In [ ]:
# A reusable helper for scheduled jobs, used here and in Parts 5 and 6.
def create_or_replace_job(conn, job_name, plsql_action, repeat_interval):
    try:
        execute_sql(conn, "BEGIN DBMS_SCHEDULER.DROP_JOB(:n, force=>TRUE); END;", {"n": job_name})
    except Exception:
        pass
    execute_sql(conn, '''BEGIN DBMS_SCHEDULER.CREATE_JOB(
        job_name => :n, job_type => 'PLSQL_BLOCK', job_action => :a,
        repeat_interval => :r, enabled => TRUE); END;''',
      {"n": job_name, "a": plsql_action, "r": repeat_interval})

# The staging queue between the in-DB producer and the OAMP consumer.
create_table(conn, '''CREATE TABLE promotion_queue (
  id RAW(16) DEFAULT SYS_GUID() PRIMARY KEY, path VARCHAR2(400), chunk CLOB,
  consumed CHAR(1) DEFAULT 'N', staged_at TIMESTAMP DEFAULT SYSTIMESTAMP)''')
print("Job helper + promotion_queue ready.")

### 4.5b The in-database producer (extract, chunk, stage)

This stored procedure is the producer half. It reads un-promoted scratch files, turns each into text, chunks
it, and stages the chunks, marking each file as staged so it is not processed twice. It runs entirely in the
engine, and we schedule it hourly.

In [ ]:
_CHUNK_PARAMS = json.dumps({"by": "words", "max": 200, "overlap": 20, "split": "recursively", "normalize": "all"})

ddl_idempotent(conn, f'''CREATE OR REPLACE PROCEDURE stage_scratch_for_promotion AS
BEGIN
  FOR f IN (SELECT path, content FROM agent_scratch WHERE promoted='N' AND is_dir='N') LOOP
    INSERT INTO promotion_queue (path, chunk)
    SELECT f.path, c.chunk
    FROM (SELECT DBMS_VECTOR_CHAIN.UTL_TO_TEXT(f.content) AS doc FROM dual) src,
         TABLE(DBMS_VECTOR_CHAIN.UTL_TO_CHUNKS(src.doc, JSON('{_CHUNK_PARAMS}'))) ct,
         JSON_TABLE(ct.column_value, '$' COLUMNS (chunk VARCHAR2(4000) PATH '$.chunk_data')) c;
    UPDATE agent_scratch SET promoted='S' WHERE path=f.path;   -- 'S' = staged
  END LOOP;
  COMMIT;
END;''')
create_or_replace_job(conn, "MEMORY_PROMOTION_JOB", "BEGIN stage_scratch_for_promotion; END;",
                      "FREQ=HOURLY;INTERVAL=1")
print("Producer ready + scheduled (MEMORY_PROMOTION_JOB, hourly).")

### 4.5c The OAMP consumer (drain the queue into long-term memory)

The consumer drains staged chunks into long-term memory through OAMP, making each one durable and searchable.
We also give the agent an immediate `promote_file_to_memory(path)` tool, for when it decides mid-task that a
specific note is worth keeping right now.

### 🧩 TODO 12 — Promote scratch files into long-term memory

The implementation in the next cell has been removed. Open **[`docs/todo12.md`](docs/todo12.md)** for a guided explanation and the solution snippet, then replace the placeholder.

Run the **`✅ TODO 12 check`** cell right after — it must pass before you continue.

In [ ]:
def drain_promotion_queue_to_oamp(limit=500):
    # 🧩 TODO 12 — consume staged chunks into OAMP long-term memory and mark them done. See docs/todo12.md
    raise NotImplementedError("Complete TODO 12 — open docs/todo12.md")

def promote_file_to_memory(path):
    execute_sql(conn, "BEGIN stage_scratch_for_promotion; END;")   # stage everything pending (incl. this file)
    n = drain_promotion_queue_to_oamp()
    return {"promoted_chunks": n, "path": path}

print("Consumer ready: drain_promotion_queue_to_oamp + promote_file_to_memory.")


In [ ]:
# ✅ TODO 12 check — do not edit. (Drives the real OAMP promotion path; may call the model for extraction.)
write_file("/promote_me.txt", "The Outdoors category drove Q3 revenue. Fulfilment lead time is the key operational risk.")
_r = promote_file_to_memory("/promote_me.txt")
assert isinstance(_r, dict) and isinstance(_r.get("promoted_chunks"), int), "promote_file_to_memory returns {'promoted_chunks': int, ...}"
_f = fetch_rows(conn, "SELECT promoted FROM agent_scratch WHERE path=:p", {"p": fs._abs("/promote_me.txt")})
assert _f and _f[0]["PROMOTED"] == "Y", "after promotion the scratch file should be marked promoted ('Y')"
print("✓ TODO 12 complete — scratch notes are promoted into long-term memory (continual learning).")


### 4.6 Checkpoint: persistence and promotion

Three proofs. First, **persistence**: we write a durable fact through OAMP, **reconnect with a fresh session**,
and recall it, which is the amnesiac problem solved. Second, the **promotion bridge**: we drop a note in
`/inbox`, run the scheduled producer plus the OAMP consumer, and then recall the note's *content* from
long-term memory by meaning, proving the scratch-to-durable bridge works end to end.

In [ ]:
# (1) durable fact survives a brand-new OAMP client on a fresh connection
mem.remember("Refunds over $500 require manager approval.")
conn_fresh = connect_with_retry(AGENT, AGENT_PWD, CFG["DSN"])
oamp_fresh = OracleAgentMemory(connection=conn_fresh, embedder=InDBOnnxEmbedder(conn_fresh, CFG["EMBED_MODEL"]),
                               llm=make_oamp_llm(),
                               extract_memories=False, table_name_prefix=CFG["OAMP_PREFIX"],
                               schema_policy="create_if_necessary")
recalled = [r.content for r in oamp_fresh.search("refund approval threshold", user_id=USER_ID, agent_id=AGENT_ID, max_results=2)]
conn_fresh.close()
print("recalled across a fresh session:", recalled)

# (2) scratch -> long-term promotion via the scheduled producer + OAMP consumer
fs.write("/inbox/supplier_note.md",
         "# Supplier risk\nNorthwind supplies 40% of Outdoors COGS; qualify a second source before Q1.")
execute_sql(conn, "BEGIN stage_scratch_for_promotion; END;")   # run the producer on demand
drained = drain_promotion_queue_to_oamp()                      # OAMP consumer
print("promoted chunks via OAMP:", drained)
print("recall promoted note:", mem.recall("who is the concentrated supplier", k=2))
print("\nPART 4 OK.")

> ### ✅ Key takeaway, Part 4
> The agent now has a **cognitive memory** with a real system of record: episodic threads, semantic facts that
> are **extracted** into clean statements from conversations, working summaries, and procedural workflow recipes that
> **track their own success and failure**. Everything is embedded in-database and survives process death, and a
> **scheduled promotion pipeline** continuously moves worth-keeping scratch notes into long-term memory through
> OAMP. The agent can now *remember*. Next, **Part 5** teaches it to *understand your data*.

---
# Part 5: The Semantic Layer

> **Where we are.** The agent can remember. But it does not yet know what *your* data means. It would have to
> guess that `net_revenue` is money, or that `orders.customer_id` joins to `customers`.
>
> **What this part does.** It turns the database's own metadata, column comments, machine-readable annotations,
> foreign keys, and even the live SQL workload, into an embedded **semantic catalog** the agent searches to
> ground itself before writing a query. It refreshes that catalog **on a schedule**, because schema meaning
> drifts.
>
> **Why it matters.** This is what separates a demo that hard-codes one query from an agent that can answer
> *new* questions over a real schema. Good metadata is, quite literally, prompt engineering for your database.

> 💡 **Definition: the semantic layer.** The **semantic layer** is the surface of the agent harness that
> *grounds* the memory, data, and information flowing through the harness in the **ground-truth, institutional
> and tribal knowledge** unique to a problem domain or execution context, what `net_revenue` actually means
> here, which joins are trustworthy, which status values are valid. It is a deliberate technique of
> **frontloading cognitive load**: you pay once, up front, to encode meaning the agent would otherwise have to
> guess (often wrongly) on every single query. Done well, the agent stops hallucinating column names and starts
> reasoning over your schema as if it had always known it.


### 5.1 A schema to reason about

We seed a small retail-commerce schema, customers, products, orders, and order lines, with deterministic fake
data so the whole notebook is self-contained. This is the world the agent will analyse and, later, automate.

In [ ]:
import random

for ddl in [
    '''CREATE TABLE customers (customer_id NUMBER PRIMARY KEY, name VARCHAR2(100), email VARCHAR2(120),
        country VARCHAR2(40), segment VARCHAR2(20), signup_date DATE)''',
    '''CREATE TABLE products (product_id NUMBER PRIMARY KEY, name VARCHAR2(100), category VARCHAR2(40),
        unit_price NUMBER(10,2), unit_cost NUMBER(10,2))''',
    '''CREATE TABLE orders (order_id NUMBER PRIMARY KEY, customer_id NUMBER REFERENCES customers,
        order_date DATE, status VARCHAR2(20), channel VARCHAR2(20))''',
    '''CREATE TABLE order_items (order_item_id NUMBER PRIMARY KEY, order_id NUMBER REFERENCES orders,
        product_id NUMBER REFERENCES products, quantity NUMBER, unit_price NUMBER(10,2),
        discount NUMBER(5,2) DEFAULT 0)''',
]:
    create_table(conn, ddl)

rnd = random.Random(42)

def seeded(table):
    return fetch_rows(conn, f"SELECT COUNT(*) n FROM {table}")[0]["N"] > 0

if not seeded("customers"):
    cats, chans, segs = ["Outdoors", "Electronics", "Home", "Apparel"], ["web", "store", "partner"], ["consumer", "smb", "enterprise"]
    execute_sql(conn, "INSERT INTO customers VALUES (:1,:2,:3,:4,:5, SYSDATE - :6)",
      [(i, f"Customer {i}", f"c{i}@example.com", rnd.choice(["US","GB","DE","FR"]),
        rnd.choice(segs), rnd.randint(30, 900)) for i in range(1, 61)], many=True)
    execute_sql(conn, "INSERT INTO products VALUES (:1,:2,:3,:4,:5)",
      [(i, f"Product {i}", rnd.choice(cats), round(rnd.uniform(10, 400), 2),
        round(rnd.uniform(5, 200), 2)) for i in range(1, 41)], many=True)
    oid, items = 1, []
    for _ in range(400):
        cust = rnd.randint(1, 60)
        execute_sql(conn, "INSERT INTO orders VALUES (:oid, :cust, SYSDATE - :age, :st, :ch)",
          {"oid": oid, "cust": cust, "age": rnd.randint(0, 180),
           "st": rnd.choice(["paid", "paid", "paid", "refunded"]), "ch": rnd.choice(chans)})
        for _ in range(rnd.randint(1, 4)):
            items.append((len(items) + 1, oid, rnd.randint(1, 40), rnd.randint(1, 5),
                          round(rnd.uniform(10, 400), 2), rnd.choice([0, 0, 0, 5, 10])))
        oid += 1
    execute_sql(conn, "INSERT INTO order_items VALUES (:1,:2,:3,:4,:5,:6)", items, many=True)
    conn.commit()
print("schema seeded. orders:", fetch_rows(conn, "SELECT COUNT(*) n FROM orders")[0]["N"],
      "| order_items:", fetch_rows(conn, "SELECT COUNT(*) n FROM order_items")[0]["N"])

### 5.2 Three ways to attach *meaning* to a schema

The agent reads meaning, not just column types. Oracle gives us several commands for attaching that meaning to
a schema; here are the ones this notebook uses, and why each is needed:

| Command | What it attaches | Why the agent needs it |
|---|---|---|
| `CREATE VIEW` | a business-named, pre-joined query (`v_revenue.net_revenue`) | the strongest grounding: encodes the *correct* joins and formulas once, so the agent never re-derives them |
| `COMMENT ON` | free-text descriptions on tables and columns | the human-readable baseline the scanner embeds, "what does this column mean?" |
| `ANNOTATIONS` | machine-readable name/value metadata on objects | structured intent (purpose, cardinality) the scanner can read programmatically |
| `CREATE DOMAIN` | a reusable value-type contract (e.g. a valid email) | the column carries its own validation rules, so meaning travels with the data |
| `V$SQL` (read) | the live SQL workload | reveals *tribal* knowledge: which columns and joins people actually use (see 5.3) |

We demonstrate the first three from oldest to newest, then scan all of them (plus the workload) uniformly:

1. **A business view** (`v_revenue`): a human-authored semantic layer that exposes clean, business-named
   columns (`net_revenue`) computed from raw tables. Views are the most powerful grounding because they encode
   the *correct* joins and formulas once.
2. **Column and table comments**: free-text descriptions, the baseline the scanner reads.
3. **Data annotations and a use-case domain**: machine-readable name and value metadata attached to schema
   objects. A *use-case domain* packages the rules for a common value type (here, a valid email) so the column
   carries its own contract.


In [ ]:
ddl_idempotent(conn, '''CREATE OR REPLACE VIEW v_revenue AS
  SELECT o.order_id, o.order_date, o.channel, c.segment, c.country, p.category, oi.quantity,
         (oi.unit_price * oi.quantity) * (1 - NVL(oi.discount,0)/100) AS net_revenue
  FROM orders o JOIN order_items oi ON oi.order_id = o.order_id
                JOIN products p     ON p.product_id = oi.product_id
                JOIN customers c    ON c.customer_id = o.customer_id
  WHERE o.status = 'paid' ''')

execute_sql(conn, "COMMENT ON COLUMN v_revenue.net_revenue IS 'Net paid revenue per line = price * qty * (1-discount)'")
execute_sql(conn, "COMMENT ON TABLE orders IS 'One row per customer order; status is paid or refunded'")

# Annotations + a use-case domain are newer features that need extra privileges, so attach them best-effort:
# if unavailable, comments alone still ground the agent and the rest of Part 5 proceeds.
try:
    ddl_idempotent(conn, '''ALTER TABLE products MODIFY (category ANNOTATIONS
      (purpose 'Product taxonomy used for revenue rollups', cardinality 'low'))''')
    ddl_idempotent(conn, '''CREATE DOMAIN IF NOT EXISTS email_dom AS VARCHAR2(120)
      CONSTRAINT email_chk CHECK (REGEXP_LIKE(email_dom, '^[^@]+@[^@]+\\.[^@]+$'))
      ANNOTATIONS (purpose 'A validated email address')''')
    ddl_idempotent(conn, "ALTER TABLE customers MODIFY (email DOMAIN email_dom)")
    print("Semantic objects created: v_revenue + comments + annotation + use-case domain.")
except Exception as e:
    print("v_revenue + comments created; annotation/domain skipped:", str(e).splitlines()[0])

### 5.3 The scanner: metadata into a searchable catalog

> 💡 **Why scan the live SQL workload (`V$SQL`), not just the static schema?** Schema meaning comes in two
> flavours. **Institutional knowledge** is written down and shared across the whole organisation: column
> comments, the `v_revenue` view, the foreign keys, the things any team can read off the schema. **Tribal
> knowledge** is the unwritten part, contained to a user, a team, or an execution context: *which* columns
> people actually filter on, *which* joins they really use, *which* query shapes recur. The live workload in
> `V$SQL` is exactly where that tribal knowledge surfaces. A mechanism that captures **both**, and refreshes on
> a schedule, is what keeps the semantic layer **fresh** and representative of how the data is really used, not
> just how it was declared.

The scanner reads the data dictionary (columns plus comments plus annotations), the **foreign keys** (joins the
agent can trust), and the **SQL workload** from `V$SQL` (the queries people actually run reveal which columns
and joins matter). Each finding becomes a short fact, written into the `semantic` namespace of our OracleVS
store, so "which columns hold revenue?" becomes a vector search. The catalog is *entity memory*: the agent's
map of the territory.

> 📎 **Why OracleVS here, and not OAMP records?** OAMP could store these schema facts (its `FactRecord` type is
> built for exactly this) and vector-search them. We keep the catalog in the OracleVS `semantic` namespace for
> one reason that protects the story we are telling: the catalog is **re-derived on a schedule by an
> in-database stored procedure** (5.4), which runs *inside* the engine and can `DELETE` + re-`INSERT` the
> catalog rows directly. OAMP is a Python client, so a `DBMS_SCHEDULER` job cannot drive it. Keeping the catalog
> in a table the stored procedure owns is what lets "the semantic layer refreshes itself, in the database" stay
> literally true.


In [ ]:
import hashlib

def scan_semantic_layer():
    facts = []
    cols = fetch_rows(conn, '''SELECT tc.table_name, tc.column_name, tc.data_type, cc.comments
        FROM user_tab_columns tc
        LEFT JOIN user_col_comments cc ON cc.table_name=tc.table_name AND cc.column_name=tc.column_name
        WHERE tc.table_name IN ('CUSTOMERS','PRODUCTS','ORDERS','ORDER_ITEMS')''')
    for c in cols:
        body = f"{c['TABLE_NAME']}.{c['COLUMN_NAME']} ({c['DATA_TYPE']})"
        if c["COMMENTS"]:
            body += f" -- {c['COMMENTS']}"
        facts.append((f"col:{c['TABLE_NAME']}.{c['COLUMN_NAME']}", body, "column"))
    try:
        for a in fetch_rows(conn, "SELECT object_name, column_name, annotation_name, annotation_value FROM user_annotations_usage"):
            facts.append((f"ann:{a['OBJECT_NAME']}", f"Annotation {a['ANNOTATION_NAME']}={a['ANNOTATION_VALUE']} "
                          f"on {a['OBJECT_NAME']}.{a.get('COLUMN_NAME')}", "annotation"))
    except Exception:
        pass
    for f in fetch_rows(conn, '''SELECT a.table_name, a.column_name, c_pk.table_name AS ref_table
        FROM user_cons_columns a
        JOIN user_constraints c ON a.constraint_name=c.constraint_name AND c.constraint_type='R'
        JOIN user_constraints c_pk ON c.r_constraint_name=c_pk.constraint_name'''):
        facts.append((f"fk:{f['TABLE_NAME']}.{f['COLUMN_NAME']}",
                      f"{f['TABLE_NAME']}.{f['COLUMN_NAME']} joins to {f['REF_TABLE']}", "fk"))
    try:
        for h in fetch_rows(conn, '''SELECT sql_text, executions FROM v$sql
            WHERE parsing_schema_name = USER AND UPPER(sql_text) LIKE 'SELECT%' AND executions > 0
            ORDER BY executions DESC FETCH FIRST 5 ROWS ONLY'''):
            key = hashlib.md5(str(h["SQL_TEXT"]).encode()).hexdigest()[:8]   # deterministic, not Python hash()
            facts.append((f"hot:{key}",
                          f"Frequently run ({h['EXECUTIONS']}x): {str(h['SQL_TEXT'])[:160]}", "workload"))
    except Exception:
        pass
    # Replace the previous catalog rows, then write the fresh ones through OracleVS.
    execute_sql(conn, f"""DELETE FROM {VSTORE_TABLE}
                          WHERE JSON_VALUE(metadata,'$.namespace')='semantic'
                            AND JSON_VALUE(metadata,'$.kind')='catalog'""")
    store.add_texts([b for _, b, _ in facts],
                    [{"namespace": "semantic", "kind": "catalog", "subject": s, "ftype": t} for s, b, t in facts])
    return len(facts)

print("semantic catalog facts embedded:", scan_semantic_layer())

### 5.4 Keeping the catalog fresh: a scheduled refresh

A one-shot scan goes stale the moment someone adds a column or rewrites a comment. This is the same
**staleness** problem we flagged for memory in Part 4, now applied to the database's own description of itself.

The fix is the same: a `DBMS_SCHEDULER` job that re-derives the catalog on a cadence. We expose the core of the
scan as a stored procedure (writing into the OracleVS table's own columns) and schedule it.

In [ ]:
ddl_idempotent(conn, f'''CREATE OR REPLACE PROCEDURE refresh_semantic_catalog AS
BEGIN
  DELETE FROM {VSTORE_TABLE}
   WHERE JSON_VALUE(metadata,'$.namespace')='semantic' AND JSON_VALUE(metadata,'$.kind')='catalog';
  INSERT INTO {VSTORE_TABLE} (id, text, metadata, embedding)
  SELECT SYS_GUID(),
         tc.table_name||'.'||tc.column_name||' ('||tc.data_type||') '||NVL('-- '||cc.comments,''),
         JSON_OBJECT('namespace' VALUE 'semantic', 'kind' VALUE 'catalog',
                     'subject' VALUE tc.table_name||'.'||tc.column_name, 'ftype' VALUE 'column'),
         VECTOR_EMBEDDING({CFG['EMBED_MODEL']} USING
            tc.table_name||'.'||tc.column_name||' '||NVL(cc.comments,'') AS DATA)
  FROM user_tab_columns tc
  LEFT JOIN user_col_comments cc ON cc.table_name=tc.table_name AND cc.column_name=tc.column_name
  WHERE tc.table_name IN ('CUSTOMERS','PRODUCTS','ORDERS','ORDER_ITEMS');
  COMMIT;
END;''')

In [ ]:
create_or_replace_job(conn, "SEMANTIC_REFRESH_JOB", "BEGIN refresh_semantic_catalog; END;",
                      "FREQ=HOURLY;INTERVAL=6")
print("Scheduled SEMANTIC_REFRESH_JOB (every 6h).")

### 5.5 Checkpoint: grounding works, and stays fresh

We ask which objects hold a revenue signal (and get them back by meaning), then change a comment, run the
refresh procedure, and confirm the newly described column is now findable.

First, **grounding works.** We ask "which objects hold revenue?" in plain language, and the catalog returns the
right money-bearing columns by *meaning*, not by string match. This is the agent's pre-flight check before it
writes any SQL: instead of guessing column names, it looks them up in its own map of the schema.


In [ ]:
print("Which objects hold revenue?")
for r in vector_search("which columns hold revenue or money", "semantic", 4):
    print("  -", str(r["TEXT"])[:80])


Second, **the catalog stays fresh.** To make the staleness fix from 5.4 concrete, we search for a column
*before* its meaning is described (and see the result is weak), then add a precise `COMMENT`, run the refresh
procedure, and search again, the newly-described column is now findable. Showing the before and the after is the
honest way to demonstrate that the refresh actually changed something.


In [ ]:
# Before: the discount column's meaning is not described yet, so a meaning-search for it is weak/empty.
print("BEFORE the comment, search 'percentage discount on a line':")
before_hits = vector_search("percentage discount on a line", "semantic", 2)
for r in before_hits:
    print("  -", str(r["TEXT"])[:80])
if not before_hits:
    print("  (nothing well-matched yet)")

# Add a precise comment, then re-derive the catalog on demand (what SEMANTIC_REFRESH_JOB does on a schedule).
execute_sql(conn, "COMMENT ON COLUMN order_items.discount IS 'Percentage discount applied to the line (0-100)'")
execute_sql(conn, "BEGIN refresh_semantic_catalog; END;")

print("\nAFTER the comment + refresh, the same search now finds it:")
for r in vector_search("percentage discount on a line", "semantic", 2):
    print("  -", str(r["TEXT"])[:80])
print("\nPART 5 OK.")


> ### ✅ Key takeaway, Part 5
> The agent can now **ground itself in your schema's meaning**, comments, annotations, trusted joins, and real
> workload, searched as a catalog and **refreshed on a schedule** so it tracks drift. It will stop guessing
> column names and start reasoning about your data. Everything so far has been *capabilities*. **Part 6** is
> where the agent starts **improving itself**: the toolbox, the skillbox, and standing automations.

---
# Part 6: Skills and Automations (the agent improves itself)

**Welcome to the world of continual learning.** Everything so far made the agent *capable*. This part makes it
*get better at its job over time*, without ever retraining the model, by turning the work it does into durable
assets it can recall and reuse.

> **Where we are.** The agent can remember (Part 4) and understand your schema (Part 5). It can *do* the work.
> But every session it re-derives the same SQL and re-discovers the same steps. It does not get better.
>
> **What this part does.** It closes the loop. The agent turns the work it does into durable assets held in two
> first-class registries, the **toolbox** and the **skillbox**, plus standing **automations**. The same raw
> material, a workflow it executed once, has three lifecycles:
>
> - a **workflow** is *what the agent did once* (captured automatically, with its outcome, in Part 4);
> - a **skill** is *the reusable how-to it distilled from doing something repeatedly*;
> - an **automation** is *the scheduled execution of a workflow toward a standing goal*.

> 💡 **The big idea: continual learning in token space.** A frozen model cannot update its weights. So how does
> our agent get better? It **writes skills as text** (`SKILL.md` documents), stores them in the database, and
> **retrieves the relevant ones back into its context window** when a similar task appears. The learning lives
> in *tokens it can recall*, not in *weights it cannot change*. This is continual learning in token space, and
> the database is where that learned knowledge persists. Section 6.4 makes the sharper claim that pays for all
> this effort: a promoted skill is not just *remembered*, it is *weighted more heavily* at inference than the
> raw trajectory it came from.


### 6.1 The toolbox: a tool registry retrieved by meaning

If every tool definition went into the system prompt, the prompt would bloat as the toolset grew. So we keep a
proper **tool registry**, the toolbox: a table where each row stores the tool's **name**, **description**,
**category**, its full **JSON schema** (the contract the model targets), and an **embedding** of an augmented
description (synonyms, examples, when-to-use). At run time the harness does an **HNSW search** over this table
to pull only the **top-k relevant tools** for the turn. The registry can grow to dozens of tools without
growing the prompt.

> 💡 **Two halves of a tool.** The *callable* (a Python function) cannot live in a database, so `TOOL_REGISTRY`
> keeps the functions in memory. Everything the **model** needs to *choose and call* a tool, name, description,
> JSON schema, lives in `agent_tools`, embedded and retrievable. The model selects by meaning; the harness maps
> the chosen name back to its callable.

> 📎 **Why custom DDL here, and not the `langchain_oracledb` helpers from Part 3?** In Part 3 we used the
> library's `oraclevs.create_index` because `AGENT_VSTORE` is an `OracleVS` store. The toolbox (and the
> skillbox, the workflow store, the tool-log) are **not** OracleVS stores: they carry custom typed columns (a
> tool's JSON schema, a skill's SHA and source URL, a workflow's success counters) that do not fit OracleVS's
> fixed `id / text / metadata / embedding` shape. So for these we keep our own `CREATE TABLE` and the tiny
> `create_hnsw_index` helper from Part 3. The rule of thumb: reach for the library when your store is a generic
> vector store; hand-write a typed table when you need columns the library does not model.


In [ ]:
create_table(conn, f'''CREATE TABLE agent_tools (
  name        VARCHAR2(120) PRIMARY KEY,
  description VARCHAR2(600),
  category    VARCHAR2(60),
  tool_schema JSON,
  embedding   VECTOR({CFG['VECTOR_DIM']}, FLOAT32),
  created_at  TIMESTAMP DEFAULT SYSTIMESTAMP)''')
create_hnsw_index("agent_tools")

TOOL_REGISTRY = {}   # name -> {"fn": callable, "schema": {...}}  (callables cannot live in the DB)

In [ ]:

def register_tool(name, fn, description, params_schema, *,
                  synonyms=None, examples=None, when_to_use="", when_not="", category="general"):
    schema = {"name": name, "description": description, "parameters": params_schema}
    TOOL_REGISTRY[name] = {"fn": fn, "schema": schema}
    enriched = (f"TOOL {name}: {description}\ncategory: {category}\n"
                f"synonyms: {', '.join(synonyms or [])}\nuse when: {when_to_use}\n"
                f"do not use when: {when_not}\nexamples: {' | '.join(examples or [])}")
    execute_sql(conn, f'''MERGE INTO agent_tools d USING (SELECT :n AS name FROM dual) s ON (d.name = s.name)
        WHEN MATCHED THEN UPDATE SET description=:d, category=:c, tool_schema=:sc,
            embedding = VECTOR_EMBEDDING({CFG['EMBED_MODEL']} USING :doc AS DATA)
        WHEN NOT MATCHED THEN INSERT (name, description, category, tool_schema, embedding)
            VALUES (:n, :d, :c, :sc, VECTOR_EMBEDDING({CFG['EMBED_MODEL']} USING :doc AS DATA))''',
      {"n": name, "d": description, "c": category, "sc": json.dumps(schema), "doc": enriched})

print("Toolbox ready: agent_tools table + register_tool.")

### 6.1b Retrieving from the toolbox

`retrieve_tools(query)` is an HNSW search over the toolbox that returns the most relevant tools for a turn,
their JSON schemas included. The agent loop in Part 7 uses it to bind only the tools that fit the current
question, instead of putting every tool in the prompt.

### 🧩 TODO 13 — Make tools retrievable by meaning

The implementation in the next cell has been removed. Open **[`docs/todo13.md`](docs/todo13.md)** for a guided explanation and the solution snippet, then replace the placeholder.

Run the **`✅ TODO 13 check`** cell right after — it must pass before you continue.

In [ ]:
def retrieve_tools(query, k=6):
    # 🧩 TODO 13 — HNSW search over the toolbox: rank tools by cosine distance to the query. See docs/todo13.md
    raise NotImplementedError("Complete TODO 13 — open docs/todo13.md")

print("retrieve_tools (HNSW search over the toolbox) ready.")


In [ ]:
# ✅ TODO 13 check — do not edit. This must pass before you continue.
register_tool("__check_probe_tool__", lambda: None, "a throwaway probe tool for the check", {})
_rows = retrieve_tools("__check_probe_tool__ throwaway probe", k=5)
assert isinstance(_rows, list) and _rows, "retrieve_tools should return ranked rows"
assert "NAME" in _rows[0] and "DIST" in _rows[0], "each row carries NAME + DIST (cosine distance)"
assert any(r["NAME"] == "__check_probe_tool__" for r in _rows), "the tool we just registered should be retrievable by meaning"
assert all(_rows[i]["DIST"] <= _rows[i + 1]["DIST"] for i in range(len(_rows) - 1)), "rows ordered nearest-first"
execute_sql(conn, "DELETE FROM agent_tools WHERE name='__check_probe_tool__'")  # tidy up the probe
print("✓ TODO 13 complete — tools are retrievable by meaning over the HNSW toolbox.")


### 6.2 The domain "doer" tools

The hands the agent uses on the data: explore the schema, run a **read-only, validated** SQL query, and
materialise a query as a refreshable view. `run_sql` is deliberately defensive (single statement,
`SELECT`/`WITH` only, write/DDL keywords rejected) — but treat that keyword check as a fast app-level
filter, not the boundary. The real guarantee is the **least-privilege grant from Part 1**: the `AGENT` user
was never granted the privilege to write outside its own stores, no matter what the model is talked into.

### 🧩 TODO 14 — A safe, read-only SQL tool

The implementation in the next cell has been removed. Open **[`docs/todo14.md`](docs/todo14.md)** for a guided explanation and the solution snippet, then replace the placeholder.

Run the **`✅ TODO 14 check`** cell right after — it must pass before you continue.

In [ ]:
import re as _re

def list_sources():
    rows = fetch_rows(conn, "SELECT table_name FROM user_tables WHERE table_name IN "
                            "('CUSTOMERS','PRODUCTS','ORDERS','ORDER_ITEMS') ORDER BY table_name")
    return [r["TABLE_NAME"] for r in rows] + ["V_REVENUE (view)"]

_FORBIDDEN = _re.compile(r"\b(INSERT|UPDATE|DELETE|DROP|ALTER|TRUNCATE|MERGE|GRANT|CREATE)\b", _re.I)
def run_sql(sql, max_rows=200):
    # 🧩 TODO 14 — make this a SAFE, read-only tool: ONE SELECT/WITH statement, no write/DDL, <= max_rows rows. See docs/todo14.md
    raise NotImplementedError("Complete TODO 14 — open docs/todo14.md")

def author_materialized_view(name, select_sql):
    chk = run_sql(select_sql, max_rows=1)
    if "error" in chk:
        return {"error": "select failed: " + chk["error"]}
    ddl_idempotent(conn, f"DROP MATERIALIZED VIEW {name}")
    ddl_idempotent(conn, f"CREATE MATERIALIZED VIEW {name} BUILD IMMEDIATE REFRESH COMPLETE ON DEMAND AS {select_sql}")
    return {"created": name}

print("Domain tools ready: list_sources, run_sql (read-only), author_materialized_view.")


In [ ]:
# ✅ TODO 14 check — do not edit.
_ok = run_sql("SELECT 1 AS one FROM dual")
assert "rows" in _ok and _ok["rows"], "a SELECT should return rows under the 'rows' key"
assert "error" in run_sql("DROP TABLE customers"), "a write/DDL statement must be rejected"
assert "error" in run_sql("SELECT 1 FROM dual; SELECT 2 FROM dual"), "only ONE statement is allowed"
assert "error" in run_sql("UPDATE orders SET status='x'"), "UPDATE must be rejected by the read-only guard"
print("✓ TODO 14 complete — the agent has a safe, read-only SQL tool.")


### 6.3 The skillbox: `SKILL.md` documents, versioned by SHA

A **skill** is a `SKILL.md` document — frontmatter (`name`, `description`, `tools`) plus parameterised steps
— stored server-side. Like the toolbox, `agent_skills` is a first-class table with an **embedding** for
retrieval and a **`sha`** content hash that *is* the skill's version: change the bytes, change the SHA.

### 🧩 TODO 15 — Save a SHA-versioned skill

The implementation in the next cell has been removed. Open **[`docs/todo15.md`](docs/todo15.md)** for a guided explanation and the solution snippet, then replace the placeholder.

Run the **`✅ TODO 15 check`** cell right after — it must pass before you continue.

In [ ]:
create_table(conn, f'''CREATE TABLE agent_skills (
  name VARCHAR2(120) PRIMARY KEY, description VARCHAR2(600),
  sha VARCHAR2(64), source_url VARCHAR2(600),
  skill_md CLOB, tools_used VARCHAR2(600), source_workflow_id RAW(16),
  embedding VECTOR({CFG['VECTOR_DIM']}, FLOAT32),
  created_at TIMESTAMP DEFAULT SYSTIMESTAMP, updated_at TIMESTAMP DEFAULT SYSTIMESTAMP)''')
create_hnsw_index("agent_skills")

def save_skill(name, description, skill_md, tools_used, source_workflow_id=None, source_url=None):
    # 🧩 TODO 15 — persist a SHA-versioned, embedded skill (UPSERT into agent_skills). See docs/todo15.md
    raise NotImplementedError("Complete TODO 15 — open docs/todo15.md")

print("Skillbox ready: agent_skills table + save_skill (SHA-versioned).")


In [ ]:
# ✅ TODO 15 check — do not edit.
_sha = save_skill("workshop_test_skill", "a throwaway skill for the check", "## Steps\n1. do the thing\n", ["run_sql"])
assert isinstance(_sha, str) and len(_sha) == 64, "save_skill returns the SHA-256 of the skill body (64 hex chars)"
_row = fetch_rows(conn, "SELECT sha FROM agent_skills WHERE name='workshop_test_skill'")
assert _row and _row[0]["SHA"] == _sha, "the skill should be persisted with its SHA"
execute_sql(conn, "DELETE FROM agent_skills WHERE name='workshop_test_skill'")
print("✓ TODO 15 complete — skills are written to the DB, SHA-versioned and embedded.")


### 6.3b Two-level skill retrieval (progressive disclosure)

Retrieval is **two-level**, the same progressive-disclosure pattern Anthropic uses for Agent Skills:
**Level 1** is a one-line `name: description` manifest (retrieved by HNSW each turn, so the agent cheaply
knows which skills *exist*); **Level 2** is the full `SKILL.md` body, pulled only when the agent commits to
using it. The manifest is tiny and the heavy detail arrives exactly when it carries the most weight — which
is what makes a promoted skill high-signal per token (the injection half comes in Part 7).

### 🧩 TODO 16 — Skills as searchable memory

The implementation in the next cell has been removed. Open **[`docs/todo16.md`](docs/todo16.md)** for a guided explanation and the solution snippet, then replace the placeholder.

Run the **`✅ TODO 16 check`** cell right after — it must pass before you continue.

In [ ]:
def retrieve_skills(query, k=5):                   # level 1: HNSW over the skillbox
    # 🧩 TODO 16 — rank skills by cosine distance to the query (HNSW over agent_skills). See docs/todo16.md
    raise NotImplementedError("Complete TODO 16 — open docs/todo16.md")

def build_skill_manifest(query, k=5):
    rows = retrieve_skills(query, k=k)
    return "\n".join(f"- {r['NAME']}: {r['DESCRIPTION']}" for r in rows) or "(no skills yet)"

def load_skill(name):                              # level 2: the full SKILL.md body, on demand
    # 🧩 TODO 16 — load a named skill's full SKILL.md body + metadata from agent_skills. See docs/todo16.md
    raise NotImplementedError("Complete TODO 16 — open docs/todo16.md")

print("Skill retrieval ready: manifest (L1) + load_skill (L2).")


In [ ]:
# ✅ TODO 16 check — do not edit. This must pass before you continue.
save_skill("__check_probe_skill__", "a throwaway probe skill", "## Steps\n1. probe", ["run_sql"])
_sk = retrieve_skills("__check_probe_skill__ throwaway probe", k=5)
assert isinstance(_sk, list) and _sk, "retrieve_skills should return ranked rows"
assert "NAME" in _sk[0] and "DIST" in _sk[0], "each row carries NAME + DIST (cosine distance)"
assert any(r["NAME"] == "__check_probe_skill__" for r in _sk), "the skill we just saved should be retrievable by meaning"
assert isinstance(build_skill_manifest("anything"), str), "build_skill_manifest returns a string"
assert load_skill("__no_such_skill__") == {"error": "no such skill"}, "load_skill returns an error dict for an unknown name"
assert load_skill("__check_probe_skill__")["NAME"] == "__check_probe_skill__", "load_skill returns the saved skill row"
execute_sql(conn, "DELETE FROM agent_skills WHERE name='__check_probe_skill__'")  # tidy up the probe
print("✓ TODO 16 complete — skills are searchable procedural memory (manifest + load).")


### 6.4 Promoting a workflow into a `SKILL.md` (and *why* that earns it more weight)

`promote_workflow_to_skill` takes a captured workflow and asks the LLM to distil it into a proper `SKILL.md`,
frontmatter plus a "When to use" and "Steps" body with the specifics parameterised. A deterministic fallback
guarantees a valid skill even if the model's JSON is malformed, so the pipeline never breaks.

> 💡 **Why promote at all? Because a skill is worth more to the model than the trajectory it came from.**
> A raw workflow recipe is *episodic history*: a record of what happened once, sitting as low-signal tokens
> among many in the message log, where attention is thin and a fact in the middle of a long context is the
> least likely to be used. A **skill is procedural knowledge**: a distilled, named, reusable instruction the
> harness can place where the model weights it most. Two well-documented mechanisms make the same knowledge
> count for more once it is a skill:
>
> 1. **Instruction hierarchy.** Models are trained to treat instructions by *authority level*: system and
>    developer instructions outrank ordinary conversation content. A skill injected into the system prompt as an
>    approved procedure therefore carries more weight than the same steps buried in past turns. (OpenAI's
>    *Model Spec* and instruction-hierarchy work formalise exactly this ordering.)
> 2. **Attention is a budget, and position is leverage.** Effective context engineering is about spending a
>    *limited attention budget* on the *smallest set of high-signal tokens*. A short, curated skill is far
>    higher-signal per token than a verbose trajectory, and placing it near the top of the prompt counters the
>    "lost in the middle" decay that buries mid-context history.
>
> This is the engine of *continual learning in token space*: we are not changing the model's weights, we are
> changing **how much weight a piece of knowledge gets at inference** by moving it from low-authority history
> into high-authority, high-signal instructions. The retrieval design in 6.3 (cheap discovery, full body only
> on demand) and the injection in Part 7 (skills first, framed as authoritative) are both built to cash in on
> this. *Sources: Anthropic, "Equipping agents with Agent Skills" and "Effective context engineering for AI
> agents"; OpenAI, the Model Spec instruction hierarchy.*

Crucially, `promote_workflow_to_skill` then sets the workflow's `promoted='Y'`. Because `recall_workflow` only
returns un-promoted rows (Part 4), the raw recipe **drops out of context retrieval** from now on: the skill is
the canonical, higher-weight version, and we do not want the agent to spend attention on both.

#### The promote-workflow-to-skill flow

The pipeline that turns a one-off trajectory into a reusable, higher-weight skill:

```mermaid
flowchart TB
    WF["Captured workflow recipe<br/>(intent · steps · tools · successes/failures)"] --> CHK{"Worth promoting?<br/>recurring AND reliable<br/>(successes > failures)"}
    CHK -->|no| KEEP["Leave as a candidate recipe<br/>(still recalled into context)"]
    CHK -->|yes| DISTIL["LLM distils it into a SKILL.md<br/>frontmatter + When-to-use + Steps"]
    DISTIL -->|"malformed JSON?"| FB["Deterministic fallback<br/>(guarantees a valid skill)"]
    DISTIL --> SAVE["save_skill: store SKILL.md + SHA + embedding<br/>in the skillbox"]
    FB --> SAVE
    SAVE --> MARK["Mark workflow promoted='Y'"]
    MARK --> RETIRE["recall_workflow now SKIPS it<br/>(the skill is the canonical version)"]
    SAVE --> INJECT["Part 7: injected FIRST in the system prompt<br/>= higher weight at inference"]
```

The two right-hand outcomes are the point: the raw recipe is **retired** from recall, and the distilled
skill is later **injected at high authority** (Part 7), so the same knowledge counts for more.

![Distilling a workflow recipe (workflow memory) into a reusable, higher-weight skill](images/workflow_to_skill.png)

In [ ]:
def _skill_md(name, description, tools, body):
    return (f"---\nname: {name}\ndescription: {description}\ntools: {', '.join(tools)}\n---\n\n"
            f"## When to use\n{description}\n\n## Steps\n{body}\n")

### 🧩 TODO 17 — Promote a workflow into a skill (continual learning in token space)

The implementation in the next cell has been removed. Open **[`docs/todo17.md`](docs/todo17.md)** for a guided explanation and the solution snippet, then replace the placeholder.

Run the **`✅ TODO 17 check`** cell right after — it must pass before you continue.

In [ ]:
def promote_workflow_to_skill(workflow_id):
    # 🧩 TODO 17 — distil an executed workflow into a reusable SKILL.md, save it, retire the workflow. See docs/todo17.md
    raise NotImplementedError("Complete TODO 17 — open docs/todo17.md")

print("promote_workflow_to_skill ready (distils to SKILL.md, marks the workflow promoted).")


In [ ]:
# ✅ TODO 17 check — do not edit. (May call the model once to distil; falls back if unavailable.)
mem.capture_workflow("workshop check 13: revenue by category over 90 days", [{"tool": "run_sql"}], ["run_sql"], success=True)
_wid = fetch_rows(conn, "SELECT id FROM agent_workflow WHERE intent LIKE 'workshop check 13%' FETCH FIRST 1 ROWS ONLY")[0]["ID"]
_res = promote_workflow_to_skill(_wid)
assert isinstance(_res, dict) and "promoted_skill" in _res, "promote_workflow_to_skill returns {'promoted_skill': name, ...}"
_row = fetch_rows(conn, "SELECT skill_md FROM agent_skills WHERE name=:n", {"n": _res["promoted_skill"]})
assert _row and _row[0]["SKILL_MD"] and "##" in _row[0]["SKILL_MD"], "a SKILL.md (markdown) was written for the promoted skill"
_wf = fetch_rows(conn, "SELECT promoted FROM agent_workflow WHERE id=:i", {"i": _wid})
assert _wf and _wf[0]["PROMOTED"] == "Y", "the source workflow should be marked promoted ('Y') so recall retires it"
execute_sql(conn, "DELETE FROM agent_skills WHERE source_workflow_id=:i", {"i": _wid})  # tidy up the check's rows
execute_sql(conn, "DELETE FROM agent_workflow WHERE id=:i", {"i": _wid})
print("✓ TODO 17 complete — a repeated workflow distils into a reusable SKILL.md.")


### 6.5 The harvester: promote what recurs *and* works

`harvest_skills()` makes promotion automatic: it scans workflow memory and promotes only candidates that
**recur** (≥ `min_occurrences`), are **recent**, and are **reliable** (more successes than failures) — we
don't enshrine a workflow that keeps failing. Because distillation calls the LLM, it runs in the harness;
you'd invoke it on a cadence, like the promotion consumer from Part 4.

### 🧩 TODO 18 — Harvest recurring workflows into skills

The implementation in the next cell has been removed. Open **[`docs/todo18.md`](docs/todo18.md)** for a guided explanation and the solution snippet, then replace the placeholder.

Run the **`✅ TODO 18 check`** cell right after — it must pass before you continue.

In [ ]:
HARVEST_KNOBS = {"min_occurrences": 3, "recency_days": 30}

def harvest_skills():
    # 🧩 TODO 18 — promote only recurring, recent, reliable workflows into skills. See docs/todo18.md
    raise NotImplementedError("Complete TODO 18 — open docs/todo18.md")

print("Harvester ready: harvest_skills() promotes only recurring, reliable workflows.")


In [ ]:
# ✅ TODO 18 check — do not edit.
_h = harvest_skills()
assert isinstance(_h, list), "harvest_skills returns a list of promoted skill names (empty if nothing qualifies yet)"
print("✓ TODO 18 complete — recurring, reliable workflows are auto-harvested into skills:", _h or "(none qualify yet)")


### 6.6 Automations: a proven workflow, scheduled

An **automation** promotes a proven `SELECT` into a standing deliverable: it materialises the query as a view
and schedules its refresh, then registers it. The build steps are captured back into workflow memory (marked a
**success**), so the *next* automation the agent builds recalls this recipe instead of re-deriving it.

> 💡 **Key term: materialized view.** A plain view is just a saved query that re-runs every time you read it. A
> *materialized view* stores the query's **results** as a real table and recomputes them on a schedule (here,
> via `DBMS_MVIEW.REFRESH`). That is exactly what a recurring report wants: the numbers are pre-computed and
> instantly readable, and a scheduler keeps them fresh.

In [ ]:
create_table(conn, '''CREATE TABLE agent_automations (
  name VARCHAR2(120) PRIMARY KEY, description VARCHAR2(600), artifact VARCHAR2(120),
  job_name VARCHAR2(120), cadence_hours NUMBER, select_sql CLOB, created_at TIMESTAMP DEFAULT SYSTIMESTAMP)''')

def create_automation(name, select_sql, cadence_hours=24, description=""):
    # The model may pass cadence_hours as 24.0 or "24"; coerce to a clean int (INTERVAL=24.0 is rejected).
    try:
        cadence_hours = int(float(cadence_hours))
    except Exception:
        cadence_hours = 24
    name = _re.sub(r"[^A-Za-z0-9_]", "_", name).strip("_").lower() or "automation"
    mv, job = f"MV_{name.upper()}", f"AUTO_{name.upper()}_JOB"
    res = author_materialized_view(mv, select_sql)
    if "error" in res:
        mem.capture_workflow(f"build automation {name}: {description}", [{"step": "author_select"}],
                             ["author_materialized_view", "create_automation"], success=False)
        return res
    create_or_replace_job(conn, job, f"BEGIN DBMS_MVIEW.REFRESH('{mv}', 'C'); END;",
                          f"FREQ=HOURLY;INTERVAL={cadence_hours}")
    execute_sql(conn, "DELETE FROM agent_automations WHERE name=:n", {"n": name})
    execute_sql(conn, '''INSERT INTO agent_automations (name, description, artifact, job_name, cadence_hours, select_sql)
               VALUES (:n,:d,:a,:j,:c,:s)''',
      {"n": name, "d": description, "a": mv, "j": job, "c": cadence_hours, "s": select_sql})
    mem.capture_workflow(f"build automation {name}: {description}",
                         [{"step": "author_select"}, {"step": "materialize", "mv": mv}, {"step": "schedule", "job": job}],
                         ["author_materialized_view", "create_automation"], success=True)
    return {"automation": name, "artifact": mv, "job": job, "cadence_hours": cadence_hours}

def list_automations():
    return fetch_rows(conn, "SELECT name, artifact, job_name, cadence_hours, description FROM agent_automations ORDER BY created_at")

print("create_automation + list_automations ready.")

### 6.7 Registering the full toolset

Now we register every tool: the **file tools** from Part 2 (so the agent can use its scratchpad as short-term
memory), the domain doers, the memory tools, the skill tools, and the automation tools. Each is registered with augmented metadata so semantic retrieval finds it by *intent*.

In [ ]:
# Scratchpad / short-term memory tools (Part 2)
register_tool("write_file", lambda path, content: write_file(path, content),
              "Write or overwrite a scratch file (short-term memory).", {"path": "string", "content": "string"},
              synonyms=["save note", "jot", "scratch"], when_to_use="to stash an intermediate result", category="files")
register_tool("read_file", lambda path: read_file(path),
              "Read a scratch file's contents.", {"path": "string"}, category="files")
register_tool("grep_files", lambda pattern: grep_files(pattern),
              "Search scratch files for a pattern.", {"pattern": "string"}, category="files")
register_tool("list_files", lambda: list_files(),
              "List scratch files.", {}, category="files")
# Domain doers
register_tool("list_sources", lambda: list_sources(),
              "List queryable tables/views in the analytics schema.", {},
              synonyms=["schema", "tables", "what data"], when_to_use="before writing SQL", category="explore")
register_tool("run_sql", lambda sql: run_sql(sql),
              "Run a read-only SELECT and return rows.", {"sql": "string"},
              synonyms=["query", "select", "analyze"], when_to_use="to answer an analytical question", category="data")
register_tool("create_automation", lambda name, select_sql, cadence_hours=24, description="":
              create_automation(name, select_sql, cadence_hours, description),
              "Promote a SELECT into a scheduled, refreshed materialized view.",
              {"name": "string", "select_sql": "string", "cadence_hours": "number", "description": "string"},
              synonyms=["schedule", "daily report", "recurring"], category="automation")
register_tool("list_automations", lambda: list_automations(),
              "List existing automations.", {}, category="automation")
# Memory tools
register_tool("search_memory", lambda query: mem.recall(query),
              "Search durable long-term memory by meaning.", {"query": "string"}, category="memory")
register_tool("remember_fact", lambda fact: (mem.remember(fact), {"saved": fact[:60]})[1],
              "Save a durable fact to long-term memory.", {"fact": "string"}, category="memory")
register_tool("recall_workflow", lambda query: mem.recall_workflow(query),
              "Recall a proven workflow recipe before building.", {"query": "string"}, category="memory")
register_tool("promote_file_to_memory", lambda path: promote_file_to_memory(path),
              "Promote a scratch file into durable long-term memory.", {"path": "string"}, category="memory")
# Skill tools
register_tool("find_skill", lambda query: build_skill_manifest(query),
              "List relevant skills (level-1 manifest).", {"query": "string"}, category="skills")
register_tool("load_skill", lambda name: load_skill(name),
              "Load a skill's full SKILL.md body (level-2).", {"name": "string"}, category="skills")
print("Registered tools:", list(TOOL_REGISTRY.keys()))

### 6.8 Checkpoint: capture, promote, harvest, automate

We exercise the whole lifecycle: capture a workflow, distil it into a `SKILL.md` and see it appear in the
manifest (and confirm the raw workflow is now retired from recall), run the harvester over seeded *successful*
repeats, and build a real automation (a materialized view, a scheduler job, and a registry row).

In [ ]:
wid, _ = mem.capture_workflow("compute revenue by product category for last 90 days",
                              [{"step": "list_sources"}, {"step": "run_sql"}], ["list_sources", "run_sql"], success=True)
some_wf = fetch_rows(conn, "SELECT id FROM agent_workflow ORDER BY created_at FETCH FIRST 1 ROWS ONLY")[0]["ID"]

promoted = promote_workflow_to_skill(some_wf)

print("promoted skill:", promoted.get("promoted_skill"))
print("recall_workflow excludes the promoted recipe now:",
      [r["INTENT"][:40] for r in mem.recall_workflow("revenue by category")])
print("manifest for 'category revenue':\n", build_skill_manifest("revenue by category"))

for _ in range(3):
    mem.capture_workflow("monthly active customers by segment", [{"step": "run_sql"}], ["run_sql"], success=True)
print("harvested:", harvest_skills())

auto = create_automation("revenue_by_category",
    "SELECT category, ROUND(SUM(net_revenue),2) AS revenue FROM v_revenue "
    "WHERE order_date >= SYSDATE-90 GROUP BY category", cadence_hours=24,
    description="Daily revenue by product category (90d)")
print("automation:", auto)
print("\nPART 6 OK.")

### 6.9 The registries in action

A focused look at the toolbox: **tool retrieval returns the right tools by meaning, with their JSON schemas
straight from the database** — exactly what the agent loop binds to the model each turn.

In [ ]:
# (1) Toolbox: retrieve the right tools by meaning; their JSON schemas come from the DB.
print("tools for 'schedule a result to refresh daily':")
for h in retrieve_tools("schedule a result to refresh daily", k=3):
    schema = h["TOOL_SCHEMA"] or {}
    print(f"   - {h['NAME']}  params={list(schema.get('parameters', {}).keys())}")


> ### ✅ Key takeaway, Part 6
> The agent now **improves itself**, and its capabilities live in **first-class registries** searched by HNSW:
> the **toolbox** (`agent_tools`, each tool's JSON schema plus embedding) and the **skillbox** (`agent_skills`,
> `SKILL.md` plus SHA plus source URL plus embedding). It distils the work it repeats into skills pulled back
> into context on demand (continual learning in token space), retires the raw recipe once a skill exists, keeps
> externally sourced skills in sync by SHA, harvests only the workflows that actually succeed, and can gate any
> automation behind a human. Everything is in place except the thing that *drives* it. **Part 7** builds the
> loop.

---
# Part 7: The Agent Loop

> **Where we are.** We have memory, grounding, the toolbox, the skillbox, and automations. Every one of them is
> inert until something *drives* the model: assembling context, calling the model, running the tools it asks
> for, persisting what happened, and deciding whether to go again.
>
> **What this part does.** It builds that driver as a **typed state graph**. The graph gives us explicit state,
> conditional routing, checkpointing, and a budget as the stop condition. It **recalls skills and proven
> workflow recipes dynamically into context every turn**, and it **recovers from transient failures**.
>
> **Why it matters.** The loop is the harness component that turns a frozen model into an *agent*, and it is
> where every layer below converges into one prompt.

![The agent loop — the typed graph (assemble_context → call_model → dispatch_tools → persist) that drives every layer](images/agent_loop.png)

### 7.1 Exposing the toolbox to the model

The model calls tools by name, so each registered tool needs a schema the model can target. We build a real
argument schema for each tool from its `params`, and wrap the underlying function so its result comes back as a
JSON string the model can read.

In [ ]:
import time as _time
from typing import TypedDict, List, Any, Dict
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import StructuredTool
from pydantic import create_model

_TYPEMAP = {"string": str, "number": float, "integer": int, "boolean": bool}

def _as_lc_tool(name, spec):
    params = spec["schema"]["parameters"]
    args_schema = None
    if params:
        args_schema = create_model(name + "_Args", **{k: (_TYPEMAP.get(v, str), ...) for k, v in params.items()})
    def _runner(**kwargs):
        return json.dumps(spec["fn"](**kwargs), default=str)[:4000]
    return StructuredTool.from_function(func=_runner, name=name,
                                        description=spec["schema"]["description"], args_schema=args_schema)

ALL_LC_TOOLS = {n: _as_lc_tool(n, s) for n, s in TOOL_REGISTRY.items()}
print("Tool adapters ready:", list(ALL_LC_TOOLS.keys()))

### 7.2 The state and the four nodes

> 💡 **Key term: state graph.** A *state graph* (the abstraction LangGraph gives us) is a set of named steps
> called **nodes**, connected by **edges** that decide what runs next, all sharing one **typed state** object
> that each node reads and updates. It beats a hand-rolled `while` loop because the state is explicit, the
> routing is declared rather than tangled in `if`s, and the whole thing can be checkpointed and resumed. Our
> graph has four nodes and one routing edge.

`AgentState` is the typed memory of one run. The four nodes are the verbs of the loop:

- **`assemble_context`**: the heart of the harness. It rebuilds the prompt each turn from four sources: the
  **schema catalog** (Part 5), the **skill manifest** (Part 6), the **proven workflow recipes** (Part 4,
  un-promoted only), and the **OAMP context card** (Part 4). The skill and workflow recall are *dynamic*: they
  are re-queried against the current question on every iteration, so the agent always sees the learned
  knowledge most relevant to what it is doing right now.
- **`call_model`**: bind the selected tools and invoke the model, with retry on transient failures.
- **`dispatch_tools`**: run whatever tools the model asked for, routing side-effecting tools through the human
  gate, and offloading large results (Part 8).
- **`persist`**: write the assistant's turn to episodic memory and (Part 8) maybe compact.

We define the state and tool selection first.

In [ ]:
class AgentState(TypedDict, total=False):
    messages: List[Any]
    thread_id: str
    iterations: int
    started_at: float
    card: str
    retrieved: Dict[str, Any]

def _select_tools(query, k=8):
    """HNSW search over the toolbox -> the top-k tool names for this turn, plus the always-on essentials."""
    names = [r["NAME"] for r in retrieve_tools(query, k)]
    essential = ["run_sql", "list_sources", "create_automation", "search_memory",
                 "recall_workflow", "find_skill", "load_skill", "promote_file_to_memory",
                 "write_file", "read_file"]
    return [t for t in dict.fromkeys(names + essential) if t in ALL_LC_TOOLS]

# The system prompt is the highest-authority position in the prompt (instruction hierarchy), so we use it to
# elevate skills: a promoted skill is an APPROVED procedure the model should weight above improvising.
SYSTEM = ("You are a retail-analytics agent. "
          "PROVEN SKILLS below are approved, distilled procedures: when one fits the task, follow it and load "
          "its full body with load_skill before improvising your own steps. "
          "Ground yourself in the schema catalog before writing SQL. "
          "Prefer create_automation to make a result recurring. "
          "If a tool returns an error, diagnose it and retry with a corrected call rather than giving up. "
          "Use only the provided tools, and answer concisely.")
print("State + tool selection ready.")

### 7.2a `assemble_context`: dynamic recall of skills and workflow recipes

This node is where continual learning re-enters the loop. Every iteration it queries the **skillbox** and the
**workflow recipes** against the latest user message and injects what it finds, so newly learned skills and
recently-proven paths show up automatically on the very next relevant turn. It prints a one-line trace so you
can *see* the dynamic recall happening during a run.

The four blocks it injects, in the exact order they appear in the prompt (top = highest authority and least
"lost in the middle"):

| # | Block | What it is | Where it comes from |
|---|---|---|---|
| 1 | **PROVEN SKILLS** | distilled, approved procedures the agent should prefer | the skillbox (Part 6), retrieved by meaning each turn |
| 2 | **Candidate recipes** | recent, un-promoted workflow paths worth reusing | the workflow store (Part 4), retrieved by meaning each turn |
| 3 | **Schema catalog** | the grounding facts relevant to this query | the semantic layer (Part 5), retrieved by meaning each turn |
| 4 | **Working memory** | the OAMP context card (topics, rolling summary, relevant durable memories, recent turns) | OAMP (Part 4), computed once per user turn |

Note the **ordering**, which is the implementation half of the importance argument from 6.4. Proven **skills**
go *first*, inside the system message and labelled as approved procedures, so they sit at the top of the
instruction hierarchy and at the high-attention head of the prompt. Un-promoted **candidate recipes** come
next, then schema grounding, then the working-memory card. The agent is told (in `SYSTEM`) to prefer a fitting
skill over improvising. So a workflow that earned promotion does not just persist, it is read **with more
weight** than the raw history it was distilled from.

![assemble_context — semantic catalog + skill manifest + workflow recipes + context card, re-discovered every turn](images/assemble_context_process.png)

In [ ]:
def node_assemble_context(state: AgentState):
    msgs = [m for m in state["messages"] if not isinstance(m, SystemMessage)]
    last_user = next((m.content for m in reversed(msgs) if isinstance(m, HumanMessage)), "")

    catalog = "\n".join(str(r["TEXT"]) for r in vector_search(last_user, "semantic", 5)) # Semantic layer

    manifest = build_skill_manifest(last_user, 4)                       # dynamic skill recall (skillbox)
    recipes = mem.recall_workflow(last_user, k=3)                       # dynamic recipe recall (un-promoted only)
    recipe_txt = "\n".join(
        f"- {r['INTENT']} (ran {r['OCCURRENCES']}x, {r['SUCCESSES']} ok / {r['FAILURES']} fail)" for r in recipes
    ) or "(no proven recipes yet)"
    card = state.get("card", "")

    # Order matters: PROVEN SKILLS go FIRST (top of the system prompt = highest authority and least
    # "lost in the middle" decay), candidate recipes next, then grounding and working memory. This is how
    # the harness cashes in on a skill's promotion: it places the distilled procedure where it weighs most.
    ctx = (f"# PROVEN SKILLS (approved procedures distilled from what worked; prefer these, load_skill for the body)\n{manifest}\n\n"
           f"# CANDIDATE WORKFLOW RECIPES (recent paths, not yet promoted to skills)\n{recipe_txt}\n\n"
           f"# SCHEMA (semantic catalog)\n{catalog}\n\n"
           f"# WORKING MEMORY (OAMP context card)\n{card}")
    state["retrieved"] = {"tools": _select_tools(last_user), "n_recipes": len(recipes)}
    state["messages"] = [SystemMessage(content=SYSTEM + "\n\n" + ctx)] + msgs
    print(f"  [assemble] injected skills-first context + {len(recipes)} candidate recipe(s) dynamically")
    return state

print("node_assemble_context ready (dynamic skill + workflow recall).")

### 7.2b `call_model`: invoke with failure recovery

A naive loop crashes the moment a model call hiccups (a rate limit, a dropped connection). Production harnesses
do not. `_invoke_with_retry` retries transient failures with exponential backoff, and if the model is still
unreachable after several tries, the node degrades gracefully by appending a plain answer that ends the run
rather than throwing. This is the failure-recovery half of harness engineering.

In [ ]:
def _invoke_with_retry(model, messages, attempts=3, base_delay=1.0):
    delay = base_delay
    for i in range(attempts):
        try:
            return model.invoke(messages)
        except Exception as e:
            if i == attempts - 1:
                raise
            print(f"  [recover] model call failed ({type(e).__name__}); retry {i+1}/{attempts} in {delay:.0f}s")
            _time.sleep(delay); delay *= 2

def node_call_model(state: AgentState):
    tools = [ALL_LC_TOOLS[n] for n in state["retrieved"]["tools"] if n in ALL_LC_TOOLS]
    try:
        ai = _invoke_with_retry(llm.bind_tools(tools), state["messages"])
    except Exception as e:
        print("  [recover] model unreachable after retries; ending the run gracefully:", type(e).__name__)
        ai = AIMessage(content="I could not reach the model to finish this step. Please retry.")
    state["messages"].append(ai)
    state["iterations"] = state.get("iterations", 0) + 1
    return state

print("node_call_model ready (with retry + graceful degradation).")

### 7.2c `dispatch_tools`: running the tools (and offloading)

This node runs the tools the model asked for. Large results pass through `maybe_offload` (a no-op until Part 8)
so they can leave the window. If a tool raises, the error is handed back to the model as a tool result, which
is what lets the agent self-correct.

In [ ]:
def node_dispatch_tools(state: AgentState):
    ai = state["messages"][-1]
    for call in getattr(ai, "tool_calls", []) or []:
        name, args, cid = call["name"], dict(call.get("args", {})), call.get("id")
        try:
            result = ALL_LC_TOOLS[name].invoke(args)
        except Exception as e:
            result = json.dumps({"error": str(e)})     # the model sees the error and can self-correct
        result = maybe_offload(name, result)           # Part 8 hook
        state["messages"].append(ToolMessage(content=result, tool_call_id=cid))
    return state

print("node_dispatch_tools ready (runs the model's tool calls + offload hook).")

### 7.2d `persist` and the router

`persist` writes the assistant's turn to episodic memory and (Part 8) may compact. `route` is the stop
condition: stop if we are over the iteration or wall-clock budget, otherwise loop back to run tools if the
model asked for any, else stop. A budget as the stop condition is what keeps an open-ended agent affordable.

In [ ]:
BUDGET = {"max_iterations": 8, "max_seconds": 120}

def node_persist(state: AgentState):
    ai = state["messages"][-1]
    if isinstance(ai, AIMessage) and ai.content:
        mem.add_turn(state["thread_id"], "assistant", str(ai.content))
    maybe_compact(state["thread_id"])                 # Part 8 hook
    return state

def _over_budget(state):
    return (state.get("iterations", 0) >= BUDGET["max_iterations"] or
            _time.time() - state.get("started_at", _time.time()) > BUDGET["max_seconds"])

def route(state: AgentState):
    ai = next((m for m in reversed(state["messages"]) if isinstance(m, AIMessage)), None)
    wants_tools = bool(getattr(ai, "tool_calls", None))
    if _over_budget(state):
        # Out of budget. If the model still has a tool turn open (no prose yet),
        # go summarise once so the run never ends on an empty answer.
        return "finalize" if (wants_tools or not (ai and ai.content)) else "stop"
    return "tools" if wants_tools else "stop"

def node_finalize(state: AgentState):
    """One tool-free model call to force a closing answer when the loop ran out
    of budget mid-tool-use. We strip the dangling tool_calls so the model must
    answer in prose from what it already gathered."""
    msgs = [m for m in state["messages"] if not (isinstance(m, AIMessage) and getattr(m, "tool_calls", None) and not m.content)]
    msgs = msgs + [HumanMessage(content="You are out of tool budget. Answer the user's question now "
                                        "in plain prose, using only what you've already gathered above.")]
    try:
        ai = _invoke_with_retry(llm, msgs)            # NOTE: llm, not llm.bind_tools(...)
    except Exception as e:
        ai = AIMessage(content="I gathered partial results but ran out of budget before finishing. Please retry.")
    state["messages"].append(ai)
    if isinstance(ai, AIMessage) and ai.content:
        mem.add_turn(state["thread_id"], "assistant", str(ai.content))
    print("  [finalize] budget reached -> produced a closing answer without tools")
    return state

print("node_persist + route + node_finalize + BUDGET ready.")


### 7.3 The Part-8 hooks (stubs for now)

Context engineering *modifies* this loop, so the loop calls two hooks, `maybe_offload` (in `dispatch_tools`)
and `maybe_compact` (in `persist`). We stub them as no-ops now so Part 7 runs on its own, and Part 8 redefines
them with real behaviour. The loop's *shape* will not change in Part 8; only its *behaviour* will.

In [ ]:
def maybe_offload(tool_name, result):   # replaced in Part 8
    return result
def maybe_compact(thread_id):           # replaced in Part 8
    return None
print("Part-8 hooks stubbed.")

### 7.4 Compiling the graph, with Oracle-backed checkpoints

The edges encode the control flow: assemble, call, persist, then route to either run tools and loop, or stop.
The **checkpointer** gives the run durable working state, here *in the database* via the **langgraph-oracledb**
`OracleSaver`. Every super-step is written to Oracle checkpoint tables on a dedicated connection, so a run can
be **paused, the process killed, and resumed** from the exact step it left off.

`run_agent` also closes the learning loop: after the run it records the trajectory's **outcome** (answered or
not) into workflow memory, so the agent's own success and failure history accumulates.

In [ ]:
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END
from langgraph_oracledb.checkpoint.oracle import OracleSaver

# Persist graph/working state INTO Oracle (durable + resumable), on its own connection so it never
# contends with the tool/query cursors running inside the nodes.
ckpt_conn = connect_with_retry(AGENT, AGENT_PWD, CFG["DSN"])
checkpointer = OracleSaver(ckpt_conn)
checkpointer.setup()

def build_agent_graph():
    b = StateGraph(AgentState)
    b.add_node("assemble_context", node_assemble_context)
    b.add_node("call_model", node_call_model)
    b.add_node("dispatch_tools", node_dispatch_tools)
    b.add_node("persist", node_persist)
    b.add_node("finalize", node_finalize)
    b.add_edge(START, "assemble_context")
    b.add_edge("assemble_context", "call_model")
    b.add_edge("call_model", "persist")
    b.add_conditional_edges("persist", route, {"tools": "dispatch_tools", "finalize": "finalize", "stop": END})
    b.add_edge("finalize", END)
    b.add_edge("dispatch_tools", "assemble_context")
    return b.compile(checkpointer=checkpointer)

graph = build_agent_graph()

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

#### How `run_agent` assembles a turn's context window

`run_agent` is the entry point for one user turn: it **records** the turn (`mem.add_turn`), **computes the
OAMP context card once** (`mem.context_card`, the cost discipline from Part 4), then **invokes the graph**,
whose `assemble_context` node builds the prompt each iteration. So the window comes from two halves: the
**context card** (OAMP's memory — rolling summary + relevant durable memories + recent turns) and
**`assemble_context`** (the learned half — skill manifest, workflow recipes, schema catalog, each retrieved
by meaning for *this* question). On the way out it closes the learning loop by recording the outcome.

### 🧩 TODO 19 — The agent loop

The implementation in the next cell has been removed. Open **[`docs/todo19.md`](docs/todo19.md)** for a guided explanation and the solution snippet, then replace the placeholder.

Run the **`✅ TODO 19 check`** cell right after — it must pass before you continue.

In [ ]:
def run_agent(text, thread_id="main", stream=False):
    # 🧩 TODO 19 — the agent loop: record the turn, assemble the context card, run the graph, then capture
    # the outcome. See docs/todo19.md
    raise NotImplementedError("Complete TODO 19 — open docs/todo19.md")

print("Agent graph compiled. run_agent records trajectory outcomes.")


In [ ]:
# ✅ TODO 19 check — do not edit. (Calls the model once — it may take several seconds.)
_out = run_agent("In one sentence, how many product categories are there?", thread_id="todo19-check")
assert isinstance(_out, dict) and "messages" in _out, "run_agent should return the final agent state (a dict with 'messages')"
_final = next((m.content for m in reversed(_out["messages"]) if isinstance(m, AIMessage) and m.content), "")
assert isinstance(_final, str) and _final.strip(), "the run should end with a non-empty assistant answer"
assert any(role == "user" for role, _ in mem.get_turns("todo19-check")), \
    "run_agent should persist the user turn to episodic memory (mem.add_turn)"
print("✓ TODO 19 complete — the agent loop runs end to end:", str(_final)[:80])


### 7.5 Checkpoint: a real, multi-step run

We ask an analytical question and stream the graph so you can watch the nodes fire. Notice the `[assemble]`
trace on each loop: that is the agent dynamically recalling skills and workflow recipes into context. This
single call exercises Parts 3 through 6 together, and it records its own outcome on the way out.

In [ ]:
out = run_agent("How many paid orders did we receive in the last 30 days, and what is their total value?",
                thread_id="t_part7", stream=True)

final = next((m.content for m in reversed(out["messages"]) if isinstance(m, AIMessage) and m.content), "")

print("\nAgent answer:\n", str(final)[:600])
print("\nworkflow memory now holds this run's outcome:",
      [(r["INTENT"][:45], f"{r['SUCCESSES']} ok") for r in mem.recall_workflow("recent paid orders and their value", k=2)])
print("\nPART 7 OK.")

> ### ✅ Key takeaway, Part 7
> The agent is now *alive*: a typed graph assembles grounded context (schema catalog plus a **dynamically
> recalled** skill manifest and workflow recipes plus the OAMP context card), calls the model **with retry**,
> runs the tools it asks for, persists episodic memory, records each trajectory's
> outcome, and loops under a budget. A graph beats a `while` loop for exactly the reasons that matter in
> production: typed state, checkpointing, conditional routing, resumability, recovery. One thing remains:
> keeping this affordable as sessions grow. **Part 8** wires compaction and offloading into the loop, and
> *measures* the result.

---
# Part 8: Context Engineering

> **Where we are.** The agent loop works. But a naive loop re-appends the full transcript and every full tool
> result on every turn. The window fills, and quality falls off well before the hard token limit.
>
> **What this part does.** It keeps the context window **flat** as a session grows, with two moves wired into
> the Part-7 loop: **compaction** (from the OAMP context card) and **offloading** (large tool results leave the
> window, replaced by a short reference). Then it **measures** both on real data, rather than asserting them.
>
> **Why it matters.** This is *context engineering*: treating the window as a scarce, managed resource. It is
> the difference between an agent that is sharp for three turns and one that stays sharp for three hours.

### Context rot: why a bigger window is not the fix

Three facts shape this part:

1. **Degradation is gradual and starts early.** Model quality begins to slip well before the advertised limit,
   and published studies on long-context behaviour show accuracy falling as inputs grow and as the relevant
   fact sits further from the edges of the window. Stuffing more in does not help; past a point it hurts. This
   is what people call **context rot**.
2. **Clearing tool results is the cheapest, safest compaction.** Once the model has read a tool result, the raw
   bytes rarely need to stay in the window verbatim. Keep a reference, fetch the bytes only if needed.
3. **Attention is the real budget.** What matters is not whether the context *fit*, but whether the *relevant*
   part is dense enough for the model to attend to. So you compact ahead of the limit, not at it.

> 💡 **Key terms.** *Compaction* replaces a long history with a shorter faithful summary. *Offloading* moves a
> large payload out of the window into a store and leaves a pointer. Both are forms of **structured
> note-taking**: keep terse notes in context and the bulk on disk or in tables.

### 8.1 Compaction, measured over a 30-turn session

We already built compaction in Part 4 and wired it into the loop in Part 7: `assemble_context` injects the
bounded **context card** (a summary plus a few relevant memories plus the last few turns) instead of the whole
transcript, so history is compacted by construction.

> 📎 **Is that injection claim accurate? Yes, with one nuance worth stating precisely.** `run_agent` computes
> the context card **once per user turn** (a single LLM summary call) and stores it in the run state;
> `assemble_context` then **places that same card into the prompt on every loop iteration**, alongside the
> schema catalog, the skill manifest, and the workflow recipes. So the bounded card is what the model sees in
> place of the raw transcript, which is exactly the size we measure below.

Here we **measure** it over a realistic **30-turn** session and **plot the result right in this cell**. At
several checkpoints we record two real numbers: the size of the **context card** the loop would inject, and the
size of the **full transcript** a naive loop would resend. The chart makes the two lines plain: the card
flattens out while the transcript climbs straight past it.

> 💰 We use a dedicated **non-extracting** OAMP client for this measurement only, so the turn-writes stay
> cheap, and we build the real context card at a few checkpoints across the session (it summarises with the
> LLM, so we do not need a point on every single turn). The green line is a measured artifact, not a stand-in.


In [ ]:
import matplotlib.pyplot as plt   # no Agg: we want the chart to render inline, right here

# Cheap, non-extracting client used ONLY to make this measurement affordable.
oamp_flat = OracleAgentMemory(connection=conn, embedder=InDBOnnxEmbedder(conn, CFG["EMBED_MODEL"]),
                              llm=make_oamp_llm(), extract_memories=False,
                              table_name_prefix=CFG["OAMP_PREFIX"], schema_policy="create_if_necessary")

In [ ]:
mem_flat = MemoryManager(oamp_flat, conn, USER_ID, AGENT_ID)

In [ ]:

dialogue = [
    ("user", "Walk me through what actually drove Q3 revenue, with the category breakdown and the year-on-year comparison so I can take it to the board."),
    ("assistant", "Q3 revenue rose 12% sequentially and 18% year on year. Outdoors contributed roughly half the gain on strong tent, pack, and insulation demand as the camping season peaked. Electronics added about a quarter on a new wireless audio line that sold through faster than forecast. Home was essentially flat, and Apparel slipped a few points after a soft back-to-school window we read as timing, not demand. Margin held because the growth skewed to full-price Outdoors rather than promotional Apparel."),
    ("user", "Which supplier represents the most operational risk right now, and exactly how exposed are we if something goes wrong?"),
    ("assistant", "Northwind is the concentration risk. One vendor covers roughly 40% of Outdoors cost of goods sold, so a missed shipment in peak season would hit both availability and margin, not just our negotiating leverage. We have no qualified backup that could absorb that volume on short notice today, so the exposure is structural. The board should treat this as the top operational risk for the quarter, ahead of demand softness, because the downside is a stockout in our fastest-growing category."),
    ("user", "What is the agreed mitigation, the timeline, and how will we measure that it is actually working?"),
    ("assistant", "Procurement is running a dual-sourcing trial now, aiming to qualify a second Outdoors vendor before the first quarter closes. The target is to move at least 30% of Outdoors volume to the second source by end of Q1, pulling single-vendor concentration below a third. We measure it three ways: share of Outdoors spend by vendor, lead-time variance between the two sources, and a quarterly stockout-risk score. If the trial vendor cannot hold lead times, we escalate to a third candidate."),
    ("user", "Tell me about churn. What is the single biggest driver, and how confident are we in that conclusion?"),
    ("assistant", "Fulfilment lead time is the dominant churn driver, and the relationship is sharp: once delivery slips past five days, repeat-purchase rate falls by roughly a third. We see the same pattern across every customer segment and every sales channel over the trailing four quarters, so confidence is high and it is not a seasonal artefact. Price and assortment matter at the margin, but neither explains as much variance as delivery speed. The practical implication is that logistics investment is really retention investment."),
    ("user", "Are there seasonal effects we should warn the board about before the holidays, and how large are they?"),
    ("assistant", "Yes, and the main one is Apparel returns. They spike in the two weeks after the holidays, which both compresses reported Q1 net revenue and ties up reverse-logistics capacity right when we want it free for new-season inbound. The effect is large enough to make Q1 Apparel margin look materially softer for entirely non-structural reasons, so we should pre-brief the board now. Operationally we plan to pre-stage returns handling and add temporary capacity so the wave does not delay first-quarter outbound shipments."),
    ("user", "How does channel mix affect all of this, and is any one channel growing faster than the others?"),
    ("assistant", "Web is growing fastest and now carries most Outdoors volume, which helps margin because web orders skew full-price, but it also concentrates our exposure to fulfilment lead time, since web customers are the most delivery-sensitive. Store volume is flat but high-margin and stickier, and the partner channel is lumpy and lower-margin but useful for clearing end-of-season inventory. The strategic read is that the channel driving growth is also the one most exposed to our biggest churn driver, so the two priorities reinforce each other."),
    ("user", "If we could fund only one initiative next quarter, which one would deliver the most durable value?"),
    ("assistant", "Logistics lead-time reduction, without much hesitation. It attacks the largest churn driver directly, it disproportionately helps the web channel that is driving growth, and unlike a promotion its benefit compounds because retained customers keep buying. Dual-sourcing Outdoors is a close second and is genuinely urgent on risk grounds, but it protects the downside rather than expanding the upside. For durable value rather than pure risk reduction, faster fulfilment wins, and the two can share one operational roadmap."),
    ("user", "Break down revenue by country. Where are we strong and where are we thin?"),
    ("assistant", "The US is roughly half of revenue and grows in line with the company average. Germany and the UK together are about a third and over-index on Outdoors, which is good margin but ties them to the Northwind supplier risk. France is the smallest of the four and the most price-sensitive, so it leans on promotional Apparel and carries thinner margin. No single country is a concentration risk on its own; the concentration that matters is the supplier, not the geography."),
    ("user", "How does gross margin look by category, and is the mix helping or hurting us?"),
    ("assistant", "Outdoors and Electronics carry the healthiest gross margins and are exactly where growth is concentrated, so the mix is currently helping. Home is middling and stable. Apparel is the thinnest, dragged down by discounting and by the post-holiday returns wave. The risk is that if growth rotated toward Apparel we would grow revenue while compressing margin, so we watch category mix as closely as the top line."),
    ("user", "What do inventory turns look like? Are we holding too much anywhere?"),
    ("assistant", "Outdoors turns fastest because demand is strong and we keep it lean to protect cash, which is also why the supplier risk bites: there is little buffer stock. Electronics is healthy. Home turns slowly but predictably. Apparel is the problem child, with end-of-season stock we clear through the partner channel at low margin. Overall we are not over-stocked, but Apparel's slow turns are a standing drag on working capital."),
    ("user", "Quantify the returns problem. What is it actually costing us?"),
    ("assistant", "Returns concentrate in Apparel and cost us twice: the lost sale, and the reverse-logistics handling that competes with new-season inbound. The post-holiday spike is the worst window. We have not fully separated structural returns from seasonal ones, but the seasonal wave is large enough to make Q1 Apparel margin look materially softer, which is why we want to pre-brief the board rather than explain it after the fact."),
    ("user", "Do enterprise and consumer customers behave differently in ways we should act on?"),
    ("assistant", "Yes. Enterprise and SMB customers buy larger, more predictable baskets and are far less delivery-sensitive, so they churn on service and contract terms rather than lead time. Consumer customers are the delivery-sensitive segment driving the churn signal, and they skew to the web channel. So fulfilment investment is really a consumer-retention play, while enterprise retention is an account-management play; conflating them would mis-spend the budget."),
    ("user", "Is the partner channel worth keeping, given its low margin?"),
    ("assistant", "It is, but for a specific reason: it clears end-of-season Apparel and Home stock that would otherwise sit and drag working capital. We should not grow it for its own sake, because the margin is thin and the volume is lumpy. Treat it as an inventory-clearing valve rather than a growth channel, and judge it on how cleanly it moves aging stock, not on its revenue line."),
    ("user", "Give me a grounded forecast for next quarter, with the main risks to it."),
    ("assistant", "Base case is continued growth led by Outdoors, partially offset by the seasonal Apparel returns wave that softens Q1 reported margin. The two risks that would move the number are a Northwind shipment slip, which converts directly into Outdoors stockouts in our fastest-growing category, and any further slippage in fulfilment lead time, which would deepen consumer churn. Both are exactly what the dual-sourcing and logistics initiatives are meant to de-risk."),
    ("user", "Give me the one-paragraph version I can read to the board to summarise the whole quarter."),
    ("assistant", "The quarter was strong, up 12% sequentially and 18% year on year, led by Outdoors at healthy full-price margin. Two things demand attention: supplier concentration in Outdoors, where a single vendor covers 40% of cost of goods and which we are mitigating by qualifying a second source before Q1, and fulfilment lead time, the dominant churn driver and the highest-return place to invest next quarter. Expect a softer-looking Q1 Apparel margin purely from the seasonal returns wave, which we are operationally prepared for."),
]


In [ ]:
card_series, transcript_series, turn_marks, raw_total = [], [], [], 0
flat_tid = "measure_flat"
pairs = len(dialogue) // 2
for p in range(pairs):                                    # 15 pairs == 30 turns
    for role, content in dialogue[2 * p:2 * p + 2]:
        mem_flat.add_turn(flat_tid, role, content); raw_total += len(content)
    if (p + 1) % 3 == 0 or p == pairs - 1:                # measure the card every 3rd pair (keeps LLM calls low)
        card = mem_flat.context_card(flat_tid, recent=4, relevant=3) or ""
        turns_done = 2 * (p + 1)
        card_series.append(len(card)); transcript_series.append(raw_total); turn_marks.append(turns_done)
        print(f"after {turns_done:2d} turns: context card = {len(card):>5} chars | full transcript = {raw_total:>6} chars")

In [ ]:
# Draw the two lines inline, right here.
plt.figure(figsize=(8, 4))
plt.plot(turn_marks, transcript_series, "r-o", label="context engineering OFF (full transcript)")
plt.plot(turn_marks, card_series, "g-o", label="context engineering ON (bounded context card)")
plt.xlabel("conversation turns"); plt.ylabel("context size (chars, measured)")
plt.title("Total Recall: measured flat context vs blow-up"); plt.legend(); plt.tight_layout()
plt.savefig("context_growth.png", dpi=110)
plt.show()
print("\nMeasured over 30 turns: the card stays bounded while the transcript grows past it.")

### 8.2 Offloading: get large tool results out of the window

Compaction handles the *conversation*; offloading handles *tool output*, which is where the real bloat comes
from (a SQL result or a fetched document can be huge). When a tool returns more than a threshold, we write it
out of context and hand the model a short reference instead. The routing is **substrate-aware** (the choice
from Part 2):

- **Structured, queryable results** go to a `TOOL_LOG` table, embedded so they stay *semantically searchable*.
  The model gets back `[tool_log:<id>]`.
- **Large blobs or documents** go to a file under `/scratch`, read later by progressive disclosure. The model
  gets back `[file:/scratch/...]`.

The reference filename is a **deterministic** content hash, so the same payload always maps to the same name
(no reliance on Python's per-process `hash()`). We redefine the `maybe_offload` hook the Part-7 loop already
calls; the loop's shape does not change.

In [ ]:
create_table(conn, '''CREATE TABLE agent_tool_log (
  id RAW(16) DEFAULT SYS_GUID() PRIMARY KEY, tool VARCHAR2(120), payload CLOB,
  embedding VECTOR(384, FLOAT32), created_at TIMESTAMP DEFAULT SYSTIMESTAMP)''')
create_hnsw_index("agent_tool_log", accuracy=90)

OFFLOAD_THRESHOLD = 1200   # chars


In [ ]:
def maybe_offload(tool_name, result):     # redefines the Part-7 hook
    if len(result) < OFFLOAD_THRESHOLD:
        return result
    digest = hashlib.md5(result.encode("utf-8")).hexdigest()[:10]   # deterministic, stable across runs
    if result.lstrip().startswith(("{", "[")):        # structured -> searchable table
        cur = conn.cursor()
        rid = cur.var(oracledb.DB_TYPE_RAW)
        cur.execute(f'''INSERT INTO agent_tool_log (tool, payload, embedding)
            VALUES (:t, :p, VECTOR_EMBEDDING({CFG['EMBED_MODEL']} USING :p AS DATA)) RETURNING id INTO :o''',
            {"t": tool_name, "p": result, "o": rid})
        conn.commit(); new_id = rid.getvalue()[0].hex(); cur.close()
        return f"[tool_log:{new_id}] ({len(result)} chars offloaded; searchable)"
    path = f"/tool_out/{tool_name}_{digest}.txt"      # blob -> scratch file (deterministic name)
    fs.write(path, result)
    return f"[file:{path}] ({len(result)} chars offloaded; read with read_file/grep_files)"

def fetch_tool_output(ref):
    if ref.startswith("[tool_log:"):
        rid = ref.split(":")[1].rstrip("]")
        r = fetch_rows(conn, "SELECT payload FROM agent_tool_log WHERE RAWTOHEX(id)=:i", {"i": rid.upper()})
        return r[0]["PAYLOAD"] if r else None
    if ref.startswith("[file:"):
        return read_file(ref[len("[file:"):-1])
    return None


In [ ]:

register_tool("fetch_tool_output", lambda ref: fetch_tool_output(ref),
              "Recover the full text of an offloaded tool output by its reference.",
              {"ref": "string"}, category="context")
ALL_LC_TOOLS["fetch_tool_output"] = _as_lc_tool("fetch_tool_output", TOOL_REGISTRY["fetch_tool_output"])
print("Offloading wired: TOOL_LOG table + scratch files; fetch_tool_output registered.")

### 8.2b Offloading, measured

A quick, concrete measurement: a large structured tool result is replaced in context by a short reference, and
the full payload is still recoverable on demand. This is the tool-output analogue of the conversation
compaction we measured in 8.1.

In [ ]:
big_result = json.dumps({"rows": [{"order_id": i, "category": "Outdoors", "net_revenue": round(i * 3.14, 2)}
                                  for i in range(120)]})
ref = maybe_offload("run_sql", big_result)
recovered = fetch_tool_output(ref)
print(f"full tool result : {len(big_result):>6} chars")
print(f"in-context ref   : {len(ref):>6} chars  ->  {ref[:70]}")
print(f"recoverable later: {len(recovered) if recovered else 0:>6} chars (via fetch_tool_output)")
print(f"reduction in context: {100 * (1 - len(ref)/len(big_result)):.1f}%")

### 8.3 The measured picture, in one place

The chart from 8.1, embedded here so the whole story sits in a single view: the bounded context card (green)
versus the full transcript (red) over the 30-turn session. Compaction handles the *conversation* shown here;
offloading (measured in 8.2b) does the same job for *tool output*, swapping a multi-thousand-char result for a
roughly 80-char reference. Together they keep long sessions both affordable and sharp.

In [ ]:
from IPython.display import Image
print(f"measured OFF final: {transcript_series[-1]:,} chars   |   measured ON final: {card_series[-1]:,} chars")
print("(measured values from 8.1, not a simulation; chart embedded below)")
print("\nPART 8 OK.")
Image("context_growth.png")            # display the saved chart inline as this cell's output

### 8.4 Evaluation: does memory engineering hold cost down without losing accuracy?

We have *claimed* that the OAMP context card keeps the window flat without hurting answers. This section
**evaluates** that claim head-on, with an LLM as the judge, because in harness engineering a claim you cannot
measure is just a hope.

We run the same short, four-turn analytical task three ways, differing **only** in how each turn's context is
built:

1. **`card` (memory-engineered)**: the bounded **OAMP context card**, exactly what this notebook builds.
2. **`full` (naive memory)**: re-send the **entire transcript** every turn, memory, but no compaction.
3. **`none` (no memory)**: each turn is **stateless**, with no prior context at all.

For each strategy we measure two things and judge a third:

- **Input tokens** (the operational cost), summed across the four turns from the model's own usage metadata.
- **Accuracy**, scored 0 to 1 by an **LLM-as-judge** that compares each strategy's answers against ground truth
  computed from the database.

The hypothesis, and the whole point of memory engineering: **`card` should match `full` on accuracy while
costing far fewer tokens, and should beat `none` on accuracy.** That is what "reliable operational cost without
compromising accuracy" looks like when you actually measure it.


In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# The fair test needs substantial shared prior context, so we reuse the board-meeting conversation from 8.1
# (the `dialogue` defined there). The eval questions are all answerable FROM that conversation, so the only
# thing that varies between strategies is how each one carries that history into the prompt.
SEED = dialogue
EVAL_TURNS = [
    "From our discussion, what is the single biggest driver of customer churn?",
    "Which supplier is the concentration risk, and roughly what share of Outdoors cost of goods do they cover?",
    "What is the agreed mitigation, and its first-quarter target?",
    "In one line for the board: the single most important action this quarter, and why?",
]
GROUND_TRUTH = ("The biggest churn driver is fulfilment lead time slipping past five days. The concentration risk "
                "is Northwind, about 40% of Outdoors cost of goods sold. The mitigation is to qualify a second "
                "Outdoors source and move at least 30% of volume to it by end of Q1. The single priority action "
                "is dual-sourcing the Outdoors category to cut supplier concentration risk.")

SYS_EVAL = ("You are a retail-analytics assistant. Answer each question concisely using ONLY the conversation "
            "context you are given. If you have no relevant context, say so plainly.")

def _input_tokens(resp):
    return (getattr(resp, "usage_metadata", None) or {}).get("input_tokens", 0)

def run_eval_strategy(strategy):
    """strategy in {'card','full','none'}; returns (answers, total_input_tokens)."""
    tid = f"eval_{strategy}"
    if strategy == "card":                                  # seed the card thread with the board discussion
        for role, content in SEED:
            mem_flat.add_turn(tid, role, content)
    convo = list(SEED)                                      # 'full' replays and grows this
    answers, total_in = [], 0
    for q in EVAL_TURNS:
        if strategy == "none":                              # stateless: no prior context at all
            msgs = [SystemMessage(content=SYS_EVAL), HumanMessage(content=q)]
        elif strategy == "full":                            # naive: re-send the ENTIRE transcript every turn
            msgs = [SystemMessage(content=SYS_EVAL)]
            for role, content in convo:
                msgs.append(HumanMessage(content=content) if role == "user" else AIMessage(content=content))
            msgs.append(HumanMessage(content=q))
        else:                                               # card: the bounded OAMP context card
            mem_flat.add_turn(tid, "user", q)
            card = mem_flat.context_card(tid, recent=4, relevant=3) or ""
            msgs = [SystemMessage(content=SYS_EVAL + "\n\n# CONVERSATION CONTEXT (bounded OAMP card)\n" + card),
                    HumanMessage(content=q)]
        resp = llm.invoke(msgs)
        ans = resp.content if isinstance(resp.content, str) else str(resp.content)
        total_in += _input_tokens(resp)
        answers.append(ans)
        convo += [("user", q), ("assistant", ans)]
        if strategy == "card":
            mem_flat.add_turn(tid, "assistant", ans)
    return answers, total_in

eval_results = {s: run_eval_strategy(s) for s in ["card", "full", "none"]}
for s, (ans, toks) in eval_results.items():
    print(f"[{s:5s}] total input tokens = {toks:6d} | final answer: {ans[-1][:80]}")


In [ ]:
import json as _json
import matplotlib.pyplot as plt

JUDGE_SYS = ("You are a strict grader. Given GROUND TRUTH and an assistant's four answers, score overall "
             "ACCURACY from 0.0 to 1.0: how well the answers match the ground truth AND stay consistent across "
             "the four turns. Reply with ONLY a JSON object: {\"accuracy\": <float 0-1>, \"why\": \"<one line>\"}.")

def judge_accuracy(answers):
    body = ("GROUND TRUTH:\n" + GROUND_TRUTH + "\n\nASSISTANT ANSWERS (in order):\n"
            + "\n".join(f"{i+1}. {a}" for i, a in enumerate(answers)))
    out = llm.invoke([SystemMessage(content=JUDGE_SYS), HumanMessage(content=body)]).content
    out = out if isinstance(out, str) else str(out)
    try:
        spec = _json.loads(out[out.find("{"): out.rfind("}") + 1])
        return max(0.0, min(1.0, float(spec.get("accuracy", 0.0))))
    except Exception:
        return 0.0

order = ["card", "full", "none"]
labels = {"card": "card (OAMP)", "full": "full transcript", "none": "no memory"}
colors = {"card": "#2a9d8f", "full": "#e9c46a", "none": "#e76f51"}
scores = {s: judge_accuracy(eval_results[s][0]) for s in order}
tokens = {s: eval_results[s][1] for s in order}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.bar([labels[s] for s in order], [scores[s] for s in order], color=[colors[s] for s in order])
ax1.set_title("Accuracy (LLM-as-judge)"); ax1.set_ylim(0, 1.08); ax1.set_ylabel("score (0-1)")
ax2.bar([labels[s] for s in order], [tokens[s] for s in order], color=[colors[s] for s in order])
ax2.set_title("Total input tokens (operational cost)"); ax2.set_ylabel("input tokens")
for ax, data, fmt in ((ax1, scores, "{:.2f}"), (ax2, tokens, "{:,}")):
    for i, s in enumerate(order):
        ax.text(i, data[s], fmt.format(data[s]), ha="center", va="bottom", fontsize=9)
plt.suptitle("Memory engineering: reliable cost without compromising accuracy"); plt.tight_layout()
plt.savefig("eval_results.png", dpi=110); plt.show()

print("accuracy :", {s: round(scores[s], 2) for s in order})
print("tokens   :", {s: tokens[s] for s in order})
won = "card" if scores["card"] >= max(scores["full"], scores["none"]) - 0.05 and tokens["card"] <= tokens["full"] else "?"
print(f"\nVerdict: the bounded OAMP card ('card') should match 'full' on accuracy at a fraction of the tokens, "
      f"and beat 'none'. Preferred strategy by accuracy-per-token: {won}.")
print("\nPART 8 EVAL OK.")


> ### ✅ Key takeaway, Part 8
> The window now stays **flat**, and we **measured** it: the OAMP context card keeps the conversation bounded
> while the raw transcript grows, and offloading replaces a large tool result with a short reference at a
> measured reduction of well over 90 percent, with the full payload still recoverable. The agent can run long
> without rotting. The harness is functionally complete. **Part 9** proves memory persists across sessions, and
> **Part 10** runs the whole thing as one story.

---
# Part 9: Continuity

> **Where we are.** The harness is functionally complete. One promise still needs proving before the finale:
> that the agent's memory really survives the end of a session.
>
> **What this part does.** It runs a continuity check that confirms a fact written in one session is recalled
> in a brand-new one, with no chat model involved in the recall.

### 9.1 Continuity: does the agent really remember across sessions?

The whole premise of memory engineering is persistence. We verify it directly: write a fact through OAMP in
this session, then open a **completely fresh** memory client on a new connection, a stand-in for "next week, new
process", and confirm the fact comes back. No chat model is involved in the recall; this is pure retrieval from
durable storage.

In [ ]:
mem.remember("Q1 goal: dual-source the Outdoors category to cut supplier concentration risk.")

# A brand-new connection + OAMP client = a new session entirely.
c_new = connect_with_retry(AGENT, AGENT_PWD, CFG["DSN"])
oamp_new = OracleAgentMemory(connection=c_new, embedder=InDBOnnxEmbedder(c_new, CFG["EMBED_MODEL"]),
                             llm=make_oamp_llm(),
                             extract_memories=False, table_name_prefix=CFG["OAMP_PREFIX"],
                             schema_policy="create_if_necessary")
recalled = [r.content for r in oamp_new.search("first quarter sourcing goal",
                                               user_id=USER_ID, agent_id=AGENT_ID, max_results=2)]
c_new.close()
print("recalled in a fresh session:", recalled)
assert recalled, "continuity check failed; the fact did not persist!"
print("\nPART 9 OK: memory persists across sessions.")

> ### ✅ Key takeaway, Part 9
> The continuity check proves the core promise of memory engineering: knowledge written in one session is there
> in the next, recalled from durable storage with no model in the loop. Now we put every layer together in one
> story.

---
# Part 10: The End-to-End Story

> **Where we are.** Every layer is built and checked in isolation. Now we run them as one narrative and watch
> the agent **answer**, **automate**, **learn**, and then **reuse what it learned**, getting faster the second
> time. This is the whole thesis in motion: memory makes it persistent, context engineering keeps it
> affordable, and the harness makes it act and improve. Watch for the `[assemble]` traces (dynamic recall) as it runs.

### Act 1: Answer

A fresh analytical question. The agent grounds itself in the schema catalog, retrieves the right tools, runs
SQL, and answers. The run is captured as workflow memory, with its outcome, along the way.

In [ ]:
print("=== ACT 1: answer an analytical question ===")
r1 = run_agent("Show me total revenue by product category for the last 90 days.",
               thread_id="story", stream=False)   # [assemble]/[HITL] still print from inside the nodes
a1 = next((m.content for m in reversed(r1["messages"]) if isinstance(m, AIMessage) and m.content), "")
print("\nAnswer:", str(a1)[:400])

### Act 2: Automate

The user asks to make it recurring. The agent calls `create_automation`, which builds a materialized view,
schedules its refresh, and registers it. This is a standing deliverable, not a one-off answer.

In [ ]:
print("=== ACT 2: make it a daily automation ===")
r2 = run_agent("Make that a daily automation called daily_category_revenue.",
               thread_id="story", stream=False)
print("automations:", list_automations())

### Act 3: Learn

We nudge the build pattern past the harvester's threshold with *successful* repeats, then run it. The agent
distils the repeated build-an-automation workflow into a `SKILL.md` (continual learning in token space), which
now appears in the skill manifest for future turns, and the raw workflow is retired from recall.

In [ ]:
print("=== ACT 3: harvest a repeated workflow into a skill ===")
for _ in range(3):
    mem.capture_workflow("build a scheduled revenue rollup automation",
                         [{"step": "author_materialized_view"}, {"step": "schedule"}],
                         ["author_materialized_view", "create_automation"], success=True)
print("promoted skills:", harvest_skills())
print("manifest now:\n", build_skill_manifest("build a scheduled revenue rollup"))

### Act 4: Reuse (a new session, same memory)

A new thread asks for a *second* metric. Because the skill is already in the manifest and the workflow recipe
is in memory, the agent has a proven path to follow instead of re-deriving one. Same store, new session: the
payoff of everything we built.

In [ ]:
print("=== ACT 4: do the same for a second metric, in a new session ===")
r3 = run_agent("Do the same for monthly revenue by channel; call it monthly_channel_revenue.",
               thread_id="story_2", stream=False)
print("\nautomations:", list_automations())

### The artifacts the agent created

Finally, inspect what now exists in the database that did not before the agent ran: the durable evidence that
it taught itself.

In [ ]:
print("--- scheduler jobs ---")
for j in fetch_rows(conn, "SELECT job_name, repeat_interval FROM user_scheduler_jobs ORDER BY job_name"):
    print("  ", j["JOB_NAME"], j["REPEAT_INTERVAL"])
print("--- automations registry ---");  [print("  ", a) for a in list_automations()]
print("--- materialized views ---", [m["MVIEW_NAME"] for m in fetch_rows(conn, "SELECT mview_name FROM user_mviews ORDER BY 1")])
print("--- learned skills ---")
for s in fetch_rows(conn, "SELECT name, description FROM agent_skills ORDER BY name"):
    print("  ", s["NAME"], "--", str(s["DESCRIPTION"])[:60])
print("--- offloaded tool outputs:", fetch_rows(conn, "SELECT COUNT(*) n FROM agent_tool_log")[0]["N"], "rows")
print("--- scratch files:", fs.list("/"))
print("--- long-term memories (OAMP):",
      fetch_rows(conn, f"SELECT COUNT(*) n FROM {CFG['OAMP_PREFIX']}MEMORY")[0]["N"], "rows")
print("--- see context_growth.png for the measured flat-vs-blowup chart")

## Closing: the three disciplines, demonstrated

You built an agent harness from the ground up and watched it teach itself, with every capability living inside
one database:

- **Memory engineering** made it *persistent*: episodic threads, semantic facts distilled by extraction,
  working summaries, and procedural recipes that track their own outcomes, all surviving process death because
  they live in the database, embedded in-engine.
- **Context engineering** kept it *affordable*: the OAMP context card compacted history by construction, and
  substrate-aware offloading kept tool results out of the window, both **measured**, not asserted.
- **Harness engineering** made it *act, recover, and improve*: a typed loop with retry gave
  the model hands, OracleVS and the semantic layer gave it grounding, and the workflow to skill to automation
  lifecycle turned what it did once into reusable skills and standing, scheduled deliverables.

# `Agent = Model + Harness`

The model never changed. Everything that made it useful, and everything that made it *get better*, was the
harness. And the harness lives, end to end, inside the database — with the one outbound call, the model itself, running on **Oracle's own OCI Generative AI**. The whole stack is Oracle.